# Controllo sul riconoscimento delle fasi del 04 (sessione di riferimento)

Mappe rilette dalla cache del 04 originale (bin di io.py), nessun allenamento. Generato da rep/make_fasi.py.

# 04 · Metodi "deep": CEBRA e LFADS · FALCON H1

**Obiettivo.** Nei notebook 02 e 03 ogni metodo seguiva una ricetta fissa per passare dai dati
alle coordinate: proiezioni lineari (PCA, FA, GPFA) o costruzioni sul grafo dei vicini (Isomap,
UMAP). Qui la ricetta la impara una **rete neurale**, cioè una funzione non lineare con molti
parametri regolati sui dati (review, §4.2).

- **Parte A, CEBRA** (Schneider, Lee & Mathis 2023), questo notebook.
- **Parte B, LFADS** (Pandarinath et al. 2018), §9-§15, dopo la parte A.

**CEBRA in una frase.** Una rete impara a mettere **vicini** nella mappa gli istanti che "vanno
insieme" e **lontani** quelli presi a caso. Che cosa vuol dire "vanno insieme" lo decidiamo noi:

- **CEBRA-Time**: istanti vicini nel tempo. Non usa nessuna etichetta;
- **CEBRA-Behavior**: istanti con un comportamento simile. Usa le etichette.

Due precisazioni da tenere presenti in tutto il notebook:

- **CEBRA non è un modello generativo.** Ha solo la metà che va dai dati alla mappa (l'*encoder*),
  non quella che dalla mappa ricostruisce i dati. Per questo lo chiamiamo "deep" o "contrastivo",
  mai "generativo".
- **CEBRA-Time avvicina istanti vicini nel tempo**: è proprio la "catena del tempo" che nel 03
  (§2) abbiamo dovuto evitare. Dovremo dimostrare che la mappa contiene più di quella.

Le domande:

1. Con 2-3 coordinate, CEBRA-Time riconosce le fasi del task meglio di PCA, Isomap e UMAP? E
   rispetto a un "tetto" costruito alla pari, con lo stesso input e la stessa finestra di tempo?
2. Che cosa impara: la struttura del task o solo la catena del tempo?
3. Con le etichette (CEBRA-Behavior), quanto si guadagna rispetto a un metodo lineare che usa le
   stesse etichette?

La pipeline:

```
spike_times ──(come nel 02/03: bin 20 ms, 160 canali attivi, σ = 50 ms, z-score)──► Z    12233 × 160

Z ──(§1: PCA a 25 stimata sui soli trial di training, applicata a tutta la sessione)──► input di CEBRA

input ──(CEBRA: rete che legge una finestra di tempo)──► mappa a poche dimensioni

valutazione: gli stessi 1106 punti del 03 (fasi di movimento, un bin ogni 100 ms), su trial mai visti
```

Il notebook si costruisce **un passo alla volta**: §1 dati e metro comune · §2 l'idea su un
esempio sintetico · §3 CEBRA-Time, un allenamento · §4 valutazione con il metro comune ·
§5 controlli · §6 CEBRA-Behavior · §7 sensibilità e alternative · §8 riepilogo.
**Stato attuale: completo. Parte A (CEBRA), §0-§8; parte B (LFADS), §9-§15.**

Stessa sessione di tutti i notebook (`held-in-calib`, `19250108T110520`).

**Tempo di esecuzione:** circa 6-7 minuti se gli allenamenti sono già salvati su disco (§3.1);
circa 2 ore la prima volta (molto di più con il computer sotto carico: il §7 da solo ha richiesto
quasi 2 ore). Con il computer sotto carico può volerci molto di più (un'esecuzione è
durata 39 minuti).

## 0 · Setup

In [ ]:
import os                                         # per impostare variabili d'ambiente prima degli altri import
os.environ.setdefault("KMP_WARNINGS", "off")      # nasconde un messaggio innocuo di OpenMP che comparirebbe con UMAP
os.environ.setdefault("NUMBA_THREADING_LAYER", "workqueue")  # numba (usato da UMAP) non carica una seconda copia di OpenMP: insieme a quella di scikit-learn il kernel a volte si blocca (come nel 03)
import sys                                        # per aggiungere src/ ai percorsi in cui Python cerca i moduli
import warnings                                   # per nascondere avvisi innocui
warnings.filterwarnings("ignore")                 # scikit-learn e UMAP stampano avvisi innocui (anche all'import di UMAP)
from pathlib import Path                          # percorsi dei file, indipendenti dal sistema operativo

import numpy as np                                # array e algebra lineare
import pandas as pd                               # tabelle per i riepiloghi
import matplotlib.pyplot as plt                   # figure
from joblib import Parallel, delayed              # per far girare in parallelo le ripetizioni con semi diversi
from sklearn.decomposition import PCA             # PCA (input dei metodi e metodo di confronto)
from sklearn.preprocessing import StandardScaler  # z-score colonna per colonna
from sklearn.manifold import Isomap               # Isomap, metodo di confronto (notebook 03)
from sklearn.neighbors import KNeighborsClassifier  # il classificatore "a vicini" del metro comune
from sklearn.model_selection import GroupKFold    # cross-validation che tiene insieme i bin dello stesso trial
from sklearn.metrics import balanced_accuracy_score  # accuratezza che pesa allo stesso modo tutte le fasi
from umap import UMAP                             # UMAP, metodo di confronto (notebook 03, §9)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # radice del progetto, da qualunque cartella parta il notebook
sys.path.insert(0, str(ROOT / "src"))             # rende importabile il pacchetto del progetto
from tesi_h1 import io                            # caricamento dei dati, fasi, colori (verificati passo per passo nel 02_new)

FIGS = ROOT / "figures"                           # cartella delle figure in PNG
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})  # stesso stile grafico degli altri notebook

SESS = ROOT / "data/raw/sub-HumanPitt-held-in-calib/sub-HumanPitt-held-in-calib_ses-19250108T110520.nwb"  # sessione analizzata

In [ ]:
# CONTROLLO FASI: cartelle di uscita (il 04 originale resta intatto)
FIGS = Path.home() / "tesi_h1_work/rep/fasi/figures"; FIGS.mkdir(parents=True, exist_ok=True)

In [ ]:
s = io.load_session(SESS)                                   # conteggi a 20 ms, cinematica, fasi del task
ACTIVE = io.active_units(s, min_hz=0.5)                     # i 160 canali attivi
Z = StandardScaler().fit_transform(                         # z-score per canale...
    io.smooth_rates(s.spikes, sigma_ms=50)[:, ACTIVE].astype(float))  # ...dei firing rate smussati con σ = 50 ms
n_bins = s.n_bins                                           # numero di bin della sessione

def base_tag(tag):                                          # "Presentation3" -> "Presentation": un'etichetta per tutte le presentazioni
    return "Presentation" if tag.startswith("Presentation") else tag

phase = np.array([base_tag(p) for p in io.phase_per_bin(s)], dtype=object)  # fase di ogni bin ("" se nessuna fase lo copre)
trial = s.trial_num                                         # trial di ogni bin (da 1 a 15)
MOV = np.isin(phase, io.MOVEMENT_PHASES)                    # vero per i bin che cadono in una fase di movimento

STEP = 5                                                    # un bin ogni 5 = uno ogni 100 ms (03, §2)
K_PCA = 25                                                  # dimensioni della PCA iniziale (03, §2: il numero di fattori scelto per la GPFA)
K_NN = 10                                                   # vicini del classificatore e di Isomap/UMAP (03)
bins_all = np.flatnonzero(MOV)                              # tutti i bin delle fasi di movimento
bins = bins_all[bins_all % STEP == 0]                       # i punti di valutazione: uno ogni 100 ms
FOLDS = list(GroupKFold(n_splits=5).split(bins, groups=trial[bins]))  # i 5 blocchi del 03: (indici di training, indici di test) tra i punti
TEST_TRIALS = [np.unique(trial[bins[tst]]) for _, tst in FOLDS]       # i 3 trial di test di ogni blocco

print(f"Z: {Z.shape[0]} bin × {Z.shape[1]} canali, {len(np.unique(trial))} trial, {n_bins * io.BIN_S / 60:.1f} minuti")
print(f"Punti di valutazione: {len(bins)} (bin di movimento: {len(bins_all)})")
print("Trial di test per blocco:", {k: t.tolist() for k, t in enumerate(TEST_TRIALS)})

In [ ]:
PCA_F = [PCA(K_PCA).fit(Z[bins_all[~np.isin(trial[bins_all], tt)]])  # in ogni blocco, la PCA a 25 vede solo i bin di movimento dei trial di training (03, §9.1)
         for tt in TEST_TRIALS]
X25 = [p.transform(Z) for p in PCA_F]                       # per ogni blocco, tutta la sessione proiettata sulle sue 25 direzioni (12233 × 25)
INP_PC = (bins, [(X25[k][bins[trn]], X25[k][bins[tst]], trn, tst)  # l'input del metro comune: punti, e per ogni blocco (training, test, indici)
                 for k, (trn, tst) in enumerate(FOLDS)])

def decode_map(inp, embed):                                 # accuratezza bilanciata (%) con una mappa qualsiasi, stima pulita (03, §9.1)
    b, folds = inp
    pred = np.empty(len(b), dtype=object)                   # la fase prevista per ogni punto, quando è nel test
    for Xtr, Xte, trn, tst in folds:
        Etr, Ete = embed(Xtr, Xte)                          # la mappa si stima sul training; i punti di test si proiettano
        pred[tst] = KNeighborsClassifier(n_neighbors=K_NN).fit(Etr, phase[b[trn]]).predict(Ete)
    return balanced_accuracy_score(phase[b], pred) * 100

def emb_pca(d):                                             # PCA a d dimensioni sui 25 PC di training (03, §9.1)
    def f(Xtr, Xte):
        p_ = PCA(d).fit(Xtr)
        return p_.transform(Xtr), p_.transform(Xte)
    return f

def emb_isomap(d):                                          # Isomap con k = 10 (03, §6)
    def f(Xtr, Xte):
        i_ = Isomap(n_neighbors=K_NN, n_components=d).fit(Xtr)
        return i_.embedding_, i_.transform(Xte)
    return f

def emb_umap(d, seed=0):                                    # UMAP con n_neighbors = 10, min_dist = 0.1 (03, §9.1)
    def f(Xtr, Xte):
        u = UMAP(n_neighbors=K_NN, n_components=d, min_dist=0.1, random_state=seed, n_jobs=1).fit(Xtr)
        return u.embedding_, u.transform(Xte)
    return f

def emb_none(Xtr, Xte):                                     # nessuna mappa: il classificatore lavora sulle 25 dimensioni (il tetto)
    return Xtr, Xte

SEEDS = range(5)                                            # i 5 semi delle misure che dipendono dal caso

def mean_range(a):                                          # "media (minimo-massimo)" di una serie di ripetizioni
    return f"{np.mean(a):.1f} ({np.min(a):.1f}-{np.max(a):.1f})"

REF_03 = {"PCA, d = 2": "26.0", "PCA, d = 3": "32.4", "Isomap, d = 2": "31.8", "Isomap, d = 3": "36.0",  # i numeri del 03 (§6 e §9.1), da riprodurre
          "UMAP, d = 2": "41.2 (40.4-42.3)", "UMAP, d = 3": "42.6 (41.9-43.2)", "tetto, 25 dimensioni": "42.2"}

ACC = {}                                                    # le accuratezze di questo notebook: nome -> valore (o lista di 5 valori)
for d in (2, 3):
    ACC[f"PCA, d = {d}"] = decode_map(INP_PC, emb_pca(d))
ACC["tetto, 25 dimensioni"] = decode_map(INP_PC, emb_none)

tab_ref = pd.DataFrame({"qui": {k: mean_range(v) if isinstance(v, list) else f"{v:.1f}" for k, v in ACC.items()},
                        "notebook 03": REF_03})
display(tab_ref)

In [ ]:
TRAIN_BINS = [np.flatnonzero(~np.isin(trial, tt)) for tt in TEST_TRIALS]  # per ogni blocco, tutti i bin dei 12 trial di training, in ordine di tempo
tab_bins = pd.DataFrame([{"trial di test": ", ".join(map(str, tt)),
                          "bin di training (tutti)": len(tb),
                          "di cui di movimento": int(MOV[tb].sum()),
                          "di cui di pausa": int((~MOV[tb]).sum()),
                          "punti di valutazione, training": len(trn),
                          "punti di valutazione, test": len(tst)}
                         for tt, tb, (trn, tst) in zip(TEST_TRIALS, TRAIN_BINS, FOLDS)],
                        index=pd.Index(range(5), name="blocco"))
display(tab_bins)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 15
DELTA, WIN = 10, 10

def seam_stats(tb):                                         # giunte di una sequenza di training e quanto pesano
    seams = np.flatnonzero(np.diff(tb) > 1) + 1             # posizioni nella sequenza in cui il bin precedente non è quello di 20 ms prima
    i = np.arange(len(tb) - DELTA)                          # i possibili primi elementi di una coppia (istante, istante + Δ)
    pair_cross = np.isin(i[:, None] + np.arange(1, DELTA + 1)[None, :], seams).any(1)  # la coppia scavalca una giunta?
    j = np.arange(WIN // 2, len(tb) - WIN // 2)             # i punti la cui finestra (5 bin prima, 5 dopo) sta tutta nella sequenza
    win_cross = np.isin(j[:, None] + np.arange(-WIN // 2 + 1, WIN // 2)[None, :], seams).any(1)  # la finestra contiene una giunta?
    return {"giunte": len(seams), "coppie a cavallo (%)": pair_cross.mean() * 100,
            "finestre a cavallo (%)": win_cross.mean() * 100}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 21
from scipy.ndimage import gaussian_filter1d, uniform_filter1d

T_SYN, N_CH, NOISE = 10_000, 30, 2.0

In [ ]:
# REPLICA: solo le definizioni della cella tolta 24
DELTA_SYN = 10

ref0 = 2_000

In [ ]:
from joblib.externals.loky import ProcessPoolExecutor      # un gruppo di processi separati, solo per CEBRA (diverso da quello usato da joblib per UMAP)

POOL = ProcessPoolExecutor(max_workers=2)                   # due allenamenti alla volta
CEBRA_STD = dict(model_architecture="offset10-model",       # rete convoluzionale con una finestra di 10 bin (5 prima, 5 dopo)
                 batch_size=512,                            # riferimenti per ogni passo (e altrettanti negativi)
                 learning_rate=3e-4,                        # quanto si correggono i pesi della rete a ogni passo
                 temperature=1.0,                           # τ della funzione di costo
                 time_offsets=10,                           # Δ: il positivo è 10 bin (200 ms) dopo il riferimento
                 distance="cosine",                         # somiglianza = coseno; i punti stanno su una sfera
                 output_dimension=3,                        # dimensioni della mappa
                 max_iterations=5000)                       # passi di allenamento (per i dati veri si decide al §3)

def cebra_fit(X_train, X_apply, cfg, seed, fix_seed=True, monitor=None, every=None, model_path=None):
    """Allena CEBRA-Time su X_train; restituisce la mappa di ogni array in X_apply, la funzione di costo
    a ogni passo e la durata. Gira in un processo separato (vedi sopra).
    Dal §3: monitor(passo, solver) viene chiamato ogni `every` passi; model_path = dove salvare il modello."""
    import time                                             # per misurare la durata
    import numpy as np                                      # array
    import torch                                            # la libreria delle reti neurali usata da CEBRA
    import cebra                                            # CEBRA
    import cebra.distributions.base as cdb                  # la classe che crea i generatori di numeri casuali di CEBRA (§2.7)
    torch.set_num_threads(1)                                # i calcoli pesanti sono sulla GPU: un solo thread di CPU, meno OpenMP in azione
    if not hasattr(cdb.HasGenerator, "orig_init"):          # la correzione del seme (§2.7), installata una volta per processo
        cdb.HasGenerator.orig_init = cdb.HasGenerator.__init__  # il metodo originale, che sceglie un seme a caso
        def seeded_init(self, device, seed):                # il metodo corretto
            cdb.HasGenerator.orig_init(self, device=device, seed=seed)  # fa tutto quello che faceva prima...
            if cdb.HasGenerator.fixed_seed is not None:     # ...poi, se abbiamo fissato un seme, sostituisce il generatore
                self._seed = cdb.HasGenerator.fixed_seed * 1000 + cdb.HasGenerator.counter  # un seme diverso per ogni generatore, ma sempre lo stesso
                cdb.HasGenerator.counter += 1               # il prossimo generatore avrà il seme successivo
                self._generator = torch.Generator(device=self.device).manual_seed(self._seed)
        cdb.HasGenerator.__init__ = seeded_init
    cdb.HasGenerator.fixed_seed = seed if fix_seed else None  # fix_seed=False: la libreria si comporta come in origine
    cdb.HasGenerator.counter = 0                            # i generatori di questo allenamento si numerano da 0
    torch.manual_seed(seed); np.random.seed(seed)           # semi dei pesi iniziali della rete e di numpy
    device = "mps" if torch.backends.mps.is_available() else "cpu"  # la GPU del Mac, se c'è
    model = cebra.CEBRA(**cfg, device=device, verbose=False)  # il modello con i parametri scelti
    t0 = time.time()
    if monitor is None:
        model.fit(X_train)                                  # l'allenamento: senza etichette, quindi CEBRA-Time
    else:
        model.fit(X_train, callback=monitor, callback_frequency=every)  # lo stesso, chiamando monitor(passo, solver) ogni `every` passi (§3)
        monitor(cfg["max_iterations"], model.solver_)       # e un'ultima volta alla fine
    seconds = time.time() - t0
    if model_path is not None:
        model.save(str(model_path))                         # il modello allenato, su disco (§3.1)
    out = {"maps": [model.transform(x) for x in X_apply],   # ogni sequenza proiettata da sola (bordi: si ripete il primo/ultimo bin)
           "loss": np.asarray(model.state_dict_["loss"]),   # la funzione di costo a ogni passo dell'allenamento
           "seconds": seconds, "device": device}
    if monitor is not None:
        out["monitor"] = monitor.history                    # le misure raccolte durante l'allenamento (§3)
    return out

def run_cebra(jobs, fn=None):                               # esegue una lista di allenamenti (dizionari di argomenti), due alla volta
    fn = cebra_fit if fn is None else fn                    # la funzione di allenamento (dal §6 anche quella con le etichette)
    futures = [POOL.submit(fn, **j) for j in jobs]          # li manda tutti in coda...
    return [f.result() for f in futures]                    # ...e aspetta i risultati, nello stesso ordine

In [ ]:
# REPLICA: solo le definizioni della cella tolta 28
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score

def r2_uv(E_tr, E_te):                                      # R² di u e v stimati con 10 vicini nella mappa (training -> test)
    pred = KNeighborsRegressor(n_neighbors=10).fit(E_tr, UV[:N_TR]).predict(E_te)
    return {"R² di u (lunghezza)": r2_score(UV[N_TR:, 0], pred[:, 0]), "R² di v (larghezza)": r2_score(UV[N_TR:, 1], pred[:, 1])}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 31
N_B, TAU = 512, 1.0

def logsumexp(a, axis=1):                                   # log Σ exp(a), calcolato senza superare i limiti dei numeri (si toglie il massimo)
    m = a.max(axis=axis, keepdims=True)
    return (m + np.log(np.exp(a - m).sum(axis=axis, keepdims=True))).squeeze(axis)

def infonce_library(ref_e, pos_e, neg_e, tau):              # la funzione della libreria (gira nel processo separato)
    import torch, cebra
    crit = cebra.models.criterions.FixedCosineInfoNCE(temperature=tau)
    loss, align, uniform = crit(torch.from_numpy(ref_e), torch.from_numpy(pos_e), torch.from_numpy(neg_e))
    return float(loss), float(align), float(uniform)

def bits(L, n=N_B):                                         # goodness of fit: quanto si scende sotto il livello del caso, in bit
    return (np.log(n) - L) * np.log2(np.e)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 33
rows_t = []

In [ ]:
# CONTROLLO FASI: nessun allenamento. Se una mappa non è in cache, ci si ferma invece di allenare.
def run_cebra(jobs, fn=None):
    if jobs:
        raise RuntimeError(f"{len(jobs)} allenamenti non trovati in cache: il controllo non deve allenare")
    return []

In [ ]:
import hashlib, inspect, json                               # impronte dei dati, testo del codice, salvataggio delle misure

CACHE = Path.home() / "tesi_h1_work" / "cache_04"           # dove si salvano i modelli: fuori dal Desktop (iCloud)
CACHE.mkdir(parents=True, exist_ok=True)
RETRAIN = False                                             # True = rifà tutti gli allenamenti invece di rileggerli

def job_key(job, fn=None):                                  # impronta di un allenamento: cambia se cambia qualunque cosa che lo determina
    h = hashlib.sha1()
    for f in (cebra_fit, ValMonitor.__init__, ValMonitor.__call__, ValMonitor.infonce):
        h.update(inspect.getsource(f).encode())             # il codice dell'allenamento e delle misure
    if fn is not None and fn is not cebra_fit:
        h.update(inspect.getsource(fn).encode())            # dal §6: anche il codice della funzione con le etichette
    h.update(json.dumps({k: job.get(k) for k in ("cfg", "seed", "fix_seed", "every")},
                        sort_keys=True, default=str).encode())  # parametri, seme, frequenza delle misure
    arrays = [job["X_train"], *job["X_apply"]]              # i dati...
    if job.get("monitor") is not None:
        h.update(job["monitor"].key.encode())               # ...le impostazioni delle misure di validazione...
        arrays += [s_ for seqs in job["monitor"].sets.values() for s_ in seqs]  # ...e i loro trial
    if job.get("y_train") is not None:
        arrays.append(job["y_train"])                       # dal §6: le etichette...
    for name, (Xv, yv) in sorted((job.get("val") or {}).items()):
        h.update(name.encode()); arrays += [Xv, yv]         # ...e i trial di validazione con le loro etichette
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())         # ogni numero dei dati entra nell'impronta
    return h.hexdigest()[:16]

def run_cebra_cached(jobs, fn=None):                        # come run_cebra (§2.3), ma rilegge dal disco gli allenamenti già fatti
    keys = [job_key(j, fn) for j in jobs]
    todo = [i for i, k in enumerate(keys) if RETRAIN or not (CACHE / f"{k}.npz").exists()]  # quelli da fare davvero
    fresh = run_cebra([dict(jobs[i], model_path=CACHE / f"{keys[i]}.pt") for i in todo], fn)  # due alla volta, nel processo separato
    for i, r in zip(todo, fresh):                           # i risultati nuovi si salvano subito
        np.savez(CACHE / f"{keys[i]}.npz", loss=r["loss"], seconds=r["seconds"], device=r["device"],
                 monitor=json.dumps(r.get("monitor")), extra=json.dumps(r.get("extra")),
                 **{f"map_{j}": m for j, m in enumerate(r["maps"])})
    out = []
    for k in keys:                                          # tutti i risultati, nell'ordine dei lavori, letti dal disco
        z = np.load(CACHE / f"{k}.npz")
        out.append({"maps": [z[f"map_{j}"] for j in range(sum(n.startswith("map_") for n in z.files))],
                    "loss": z["loss"], "seconds": float(z["seconds"]), "device": str(z["device"]),
                    "monitor": json.loads(str(z["monitor"])), "key": k,
                    "extra": json.loads(str(z["extra"])) if "extra" in z.files else None})  # misure in più (dal §6)
    print(f"Allenamenti: {len(todo)} eseguiti ora, {len(jobs) - len(todo)} riletti dal disco")
    return out

In [ ]:
K0 = 0                                                      # il blocco usato in questa sezione
VAL_TRIALS, VAL_TRIALS_B, CHECK_TRIALS = (12, 15), (6, 7), (4, 5)  # due coppie di trial di validazione e una di training su cui si misura (uno pari e uno dispari)
TRIALS = np.unique(trial)                                   # i 15 trial
fit_trials = [t for t in TRIALS if t not in TEST_TRIALS[K0] and t not in VAL_TRIALS]      # i 10 trial su cui CEBRA si allena (validazione 12, 15)
fit_trials_b = [t for t in TRIALS if t not in TEST_TRIALS[K0] and t not in VAL_TRIALS_B]  # i 10 trial su cui CEBRA si allena (validazione 6, 7)

def pca_seqs(fit_tr):                                       # PCA a 25 stimata sui bin di movimento dei soli trial fit_tr
    p_ = PCA(K_PCA).fit(Z[bins_all[np.isin(trial[bins_all], fit_tr)]])
    X_ = p_.transform(Z)                                    # tutta la sessione sulle sue 25 direzioni
    return p_, X_, {t: X_[trial == t].astype(np.float32) for t in TRIALS}  # e ogni trial separato, nel formato delle reti

def var_kept(p_, Xv):                                       # frazione (%) della varianza di Xv che sta nelle 25 direzioni della PCA p_
    Xc = Xv - p_.mean_
    return ((Xc @ p_.components_.T) ** 2).sum() / (Xc ** 2).sum() * 100

PCA_A, XA, SEQ_A = pca_seqs(fit_trials)                     # la PCA per la validazione con i trial 12 e 15
PCA_B, XB, SEQ_B = pca_seqs(fit_trials_b)                   # la PCA per la validazione con i trial 6 e 7

def mov(trs):                                               # i bin di movimento di alcuni trial
    return Z[bins_all[np.isin(trial[bins_all], trs)]]

display(pd.DataFrame({str(vt): {"PCA del §1.3 (12 trial: li ha visti)": var_kept(PCA_F[K0], mov(vt)),
                                "PCA sui soli 10 trial di allenamento": var_kept(p_, mov(vt))}
                      for vt, p_ in ((VAL_TRIALS, PCA_A), (VAL_TRIALS_B, PCA_B))}).T.rename_axis("trial di validazione").round(1))
print(f"Per confronto: sui 10 trial su cui è stimata, la PCA tiene il {var_kept(PCA_A, mov(fit_trials)):.1f}% della varianza; "
      f"sui trial di test del blocco 0, la PCA del §1.3 ne tiene il {var_kept(PCA_F[K0], mov(TEST_TRIALS[K0])):.1f}%")

In [ ]:
# REPLICA: solo le definizioni della cella tolta 42
class ValMonitor:
    """Durante l'allenamento misura InfoNCE (come al §2.5) su gruppi di istanti fissati in anticipo, in
    alcuni insiemi di trial proiettati uno alla volta. Viene eseguito nel processo separato."""

    def __init__(self, sets, delta=10, tau=1.0, n_batches=20, batch=512, seed=0):
        rng_m = np.random.default_rng(seed)                 # seme fisso: gli stessi gruppi a ogni misura
        self.sets, self.delta, self.tau, self.history = sets, delta, tau, []
        self.idx = {}                                       # per ogni insieme: 20 coppie (riferimenti, negativi)
        for name, seqs in sets.items():
            lens = [len(s_) for s_ in seqs]                 # lunghezza di ogni trial
            starts = np.cumsum([0] + lens[:-1])             # dove comincia ogni trial, una volta messi in fila
            refs = np.concatenate([a + np.arange(n - delta) for a, n in zip(starts, lens)])  # riferimenti con il positivo nello stesso trial
            self.idx[name] = [(rng_m.choice(refs, batch), rng_m.integers(0, sum(lens), batch)) for _ in range(n_batches)]
        self.key = f"ValMonitor({ {k: [len(s_) for s_ in v] for k, v in sets.items()} }, {delta}, {tau}, {n_batches}, {batch}, {seed})"

    def __call__(self, num_steps, solver):                  # chiamato da CEBRA ogni `every` passi
        import torch                                        # (siamo nel processo separato)
        was_training = solver.model.training                # proiettare mette la rete in modalità "uso": dopo la rimettiamo com'era
        row = {"passo": int(num_steps)}
        for name, seqs in self.sets.items():
            E = np.concatenate([solver.transform(torch.from_numpy(s_)).cpu().numpy() for s_ in seqs])  # ogni trial proiettato da solo
            row[name] = float(np.mean([self.infonce(E[r], E[r + self.delta], E[n]) for r, n in self.idx[name]]))
        solver.model.train(was_training)
        self.history.append(row)

    def infonce(self, ref, pos, neg):                       # la formula del §2.5 (coseno: i punti sono già sulla sfera)
        s_pos = (ref * pos).sum(1) / self.tau
        s_neg = ref @ neg.T / self.tau
        m = s_neg.max(1, keepdims=True)                     # si toglie il massimo per non far "esplodere" gli esponenziali
        return float(np.mean(-s_pos + m[:, 0] + np.log(np.exp(s_neg - m).sum(1))))

def rule98(v):                                              # la regola: primo passo con il 98% della discesa, arrotondato alle migliaia
    reached = v.values <= v.min() + 0.02 * (v.iloc[0] - v.min())  # dove la curva ha già fatto il 98% della discesa fino al minimo
    return int(np.ceil(v.index[np.argmax(reached)] / 1000) * 1000)

STEPS_SHOW = [0, 500, 1000, 1500, 2000, 3000, 4000, 6000, 8000, 10_000]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 47
def val_acc(E):                                             # metro comune, ma training = 10 trial, giudizio = 2 trial di validazione
    clf = KNeighborsClassifier(n_neighbors=K_NN).fit(E[ev_fit], phase[ev_fit])
    return balanced_accuracy_score(phase[ev_val], clf.predict(E[ev_val])) * 100

In [ ]:
# REPLICA: solo le definizioni della cella tolta 49
def to_lonlat(E):                                           # longitudine e latitudine di ogni punto della sfera
    return np.arctan2(E[:, 1], E[:, 0]), np.arcsin(np.clip(E[:, 2], -1, 1))

In [ ]:
N_ITER = 1000                                               # il numero di passi del §3.2

In [ ]:
from itertools import product                               # tutte le combinazioni di più elenchi

CEBRA_T = dict(CEBRA_STD, max_iterations=N_ITER)            # CEBRA-Time con i parametri del §3
OUT_DIMS = (3, 4)                                           # 3 dimensioni = 2 coordinate libere; 4 dimensioni = 3 coordinate libere
COMBOS = list(product(OUT_DIMS, SEEDS, range(5)))           # (dimensioni, seme, blocco): 50 allenamenti
res4 = run_cebra_cached([dict(X_train=X25[k][TRAIN_BINS[k]].astype(np.float32),  # il film di training del blocco k (§1.4)
                              X_apply=[X25[k][trial == t].astype(np.float32) for t in TRIALS],  # tutti i 15 trial, uno alla volta
                              cfg=dict(CEBRA_T, output_dimension=od), seed=sd)
                         for od, sd, k in COMBOS])
MAPS4 = {c: np.concatenate(r["maps"]) for c, r in zip(COMBOS, res4)}  # per ogni combinazione, la mappa di tutti i 12233 bin
print(f"Durata media di un allenamento: {np.mean([r['seconds'] for r in res4]):.0f} s")

def decode_cebra(od, sd):                                   # metro comune: in ogni blocco, punti di training e di test dalla mappa di quel blocco
    inp = (bins, [(MAPS4[(od, sd, k)][bins[trn]], MAPS4[(od, sd, k)][bins[tst]], trn, tst)
                  for k, (trn, tst) in enumerate(FOLDS)])
    return decode_map(inp, emb_none)                        # la mappa è già fatta: il classificatore lavora direttamente sulle sue coordinate

for od in OUT_DIMS:
    ACC[f"CEBRA-Time, {od} dimensioni"] = [decode_cebra(od, sd) for sd in SEEDS]  # 5 semi

E0 = MAPS4[(3, 0, 0)]; trn0, tst0 = FOLDS[0]                # controllo della questione 2, sulla mappa a 3 dimensioni (seme 0, blocco 0)
pred_euc = KNeighborsClassifier(n_neighbors=K_NN).fit(E0[bins[trn0]], phase[bins[trn0]]).predict(E0[bins[tst0]])
pred_cos = KNeighborsClassifier(n_neighbors=K_NN, metric="cosine").fit(E0[bins[trn0]], phase[bins[trn0]]).predict(E0[bins[tst0]])
print(f"Distanza in linea d'aria contro coseno: previsioni uguali per il {np.mean(pred_euc == pred_cos) * 100:.1f}% dei punti di test")

In [ ]:
# REPLICA: solo le definizioni della cella tolta 53
HALF = 5

def window_idx(b):                                          # per ogni punto, i 10 bin della sua finestra, senza uscire dal trial
    lo = np.array([first_last[trial[x]][0] for x in b])     # primo bin del trial di ogni punto
    hi = np.array([first_last[trial[x]][1] for x in b])     # ultimo bin
    return np.clip(b[:, None] + np.arange(-HALF, HALF)[None, :], lo[:, None], hi[:, None])

ROWS_4 = [("PCA, d = 2", 2), ("Isomap, d = 2", 2), ("UMAP, d = 2", 2), ("CEBRA-Time, 3 dimensioni", 2),
          ("PCA, d = 3", 3), ("Isomap, d = 3", 3), ("UMAP, d = 3", 3), ("CEBRA-Time, 4 dimensioni", 3),
          ("tetto, 25 dimensioni", 25), ("tetto alla pari: media sulla finestra (25 numeri)", 25),
          ("tetto alla pari: finestra in fila (250 numeri)", 250)]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 56
def roughness(trajs):                                       # seghettatura media (03, §6.1)
    return float(np.mean([(np.diff((T_ - T_.mean(0)) / (T_.std(0) + 1e-9), n=2, axis=0) ** 2).mean()
                          for T_ in trajs]))

def consistency(trajs, keep=None):                          # correlazione media tra ripetizioni della stessa fase (03, §6.1); keep = quali finestre usare
    keep = np.ones(len(windows), bool) if keep is None else keep
    out = []
    for p in io.MOVEMENT_PHASES:
        R = np.stack([trajs[i] for i in np.flatnonzero((WIN_TAG == p) & keep)])  # (ripetizioni, 50 bin, dimensioni)
        out.append(np.mean([np.corrcoef(R[:, :, j])[np.triu_indices(len(R), 1)].mean() for j in range(R.shape[2])]))
    return float(np.mean(out))

def traj_stats(trajs):                                      # le misure di una lista di traiettorie
    odd, even = consistency(trajs, WIN_ODD), consistency(trajs, ~WIN_ODD)
    return {"seghettatura": roughness(trajs), "consistenza, tutte le ripetizioni": consistency(trajs),
            "consistenza, solo trial dispari": odd, "consistenza, solo trial pari": even,
            "consistenza per versione (media delle due)": (odd + even) / 2}

def cut(E):                                                 # dalla mappa di tutta la sessione, le 119 finestre
    return [E[w["start"]:w["stop"]] for w in windows]

rows_t = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 57
PH_SHOW = "Release"

In [ ]:
# REPLICA: solo le definizioni della cella tolta 60
def phase_randomize(Xt, seed):                              # gemello: stesse ampiezze di Fourier, fasi casuali uguali per tutte le colonne (03, §5.2)
    rng_ = np.random.default_rng(seed)
    F = np.fft.rfft(Xt - Xt.mean(0), axis=0)                # spettro di ogni colonna
    phases = np.exp(2j * np.pi * rng_.uniform(size=F.shape[0]))  # una fase casuale per ogni frequenza
    phases[0] = 1                                           # la componente costante non si tocca
    if len(Xt) % 2 == 0:
        phases[-1] = 1                                      # nemmeno quella alla frequenza massima
    return np.fft.irfft(F * phases[:, None], n=len(Xt), axis=0) + Xt.mean(0)

def fold_inp(get):                                          # input del metro comune da una funzione blocco -> dati di tutti i bin
    return (bins, [(get(k)[bins[trn]], get(k)[bins[tst]], trn, tst) for k, (trn, tst) in enumerate(FOLDS)])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 63
from sklearn.linear_model import LinearRegression

def umap_fit(Xp, d, seed=0):                                # UMAP stimato su tutti i punti dati (03, §9.2)
    return UMAP(n_neighbors=K_NN, n_components=d, random_state=seed, n_jobs=1).fit_transform(Xp)

def run_consistency(maps):                                  # R² della regressione lineare da ogni mappa a ogni altra (coppie ordinate)
    return np.array([LinearRegression().fit(A, B).score(A, B) for i, A in enumerate(maps)
                     for j, B in enumerate(maps) if i != j])

def consistency_library(maps):                              # la funzione della libreria (nel processo separato, perché importa torch)
    import cebra
    scores, pairs, ids = cebra.sklearn.metrics.consistency_score(maps, between="runs")
    return np.asarray(scores)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 65
from scipy.spatial.distance import pdist, squareform

K_MIX, CHAIN_BINS = 10, 10

def knn_index(Xp, k=K_MIX):                                 # indici dei k vicini di ogni punto, escluso il punto stesso (03, §2)
    D = squareform(pdist(Xp)); np.fill_diagonal(D, np.inf)
    return np.argpartition(D, k, axis=1)[:, :k]

def neighbor_mix(Y, b=bins):                                # composizione dei vicini di ogni punto in una mappa Y (03, §2)
    nb = knn_index(Y)
    tr_, ph_ = trial[b], phase[b]
    chain = (tr_[nb] == tr_[:, None]) & (np.abs(b[nb] - b[:, None]) <= CHAIN_BINS)  # stesso trial, entro 200 ms
    good = ~chain & (ph_[nb] == ph_[:, None]) & (tr_[nb] != tr_[:, None])           # stessa fase, altro trial
    return {"catena del tempo (%)": chain.mean() * 100, "stessa fase, altro trial (%)": good.mean() * 100,
            "altro (%)": (1 - chain.mean() - good.mean()) * 100}

DELTAS = (1, 50)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 66
def regress_map(inp, embed, target):                        # R² con cui un regressore a 10 vicini ricava `target` dalla mappa (stima pulita, come decode_map)
    b, folds = inp
    pred = np.empty(len(b))
    for Xtr, Xte, trn, tst in folds:
        Etr, Ete = embed(Xtr, Xte)
        pred[tst] = KNeighborsRegressor(n_neighbors=K_NN).fit(Etr, target[b[trn]]).predict(Ete)
    return r2_score(target[b], pred)

ACC_OF = {"nessuna mappa, 25 dimensioni": "tetto, 25 dimensioni"}

def fmt(v, scale=1):                                        # "media (min-max)" se ci sono più semi, altrimenti il numero
    v = np.atleast_1d(v) * scale
    return mean_range(v) if len(v) > 1 else f"{v[0]:.1f}"

In [ ]:
# REPLICA: solo le definizioni della cella tolta 68
def predict_map(inp, embed):                                # le previsioni di fase di ogni punto, con la stima pulita di decode_map
    b, folds = inp
    pred = np.empty(len(b), dtype=object)
    for Xtr, Xte, trn, tst in folds:
        Etr, Ete = embed(Xtr, Xte)
        pred[tst] = KNeighborsClassifier(n_neighbors=K_NN).fit(Etr, phase[b[trn]]).predict(Ete)
    return pred

In [ ]:
# REPLICA: solo le definizioni della cella tolta 70
cats, cols = ["catena del tempo (%)", "stessa fase, altro trial (%)", "altro (%)"], ["C3", "C2", "0.82"]

In [ ]:
from sklearn.cross_decomposition import PLSRegression       # PLS: proiezione lineare guidata dalle etichette

Y_KIN = np.c_[s.kin, s.vel]                                 # le etichette: 7 posizioni + 7 velocità per ogni bin (12233 × 14)
KIN_NAMES = [f"posizione {n}" for n in io.KIN_LABELS] + [f"velocità {n}" for n in io.KIN_LABELS]

def label_z(train_trials):                                  # z-score delle 14 etichette con media e deviazione dei soli trial di training
    m = np.isin(trial, train_trials)
    return ((Y_KIN - Y_KIN[m].mean(0)) / Y_KIN[m].std(0)).astype(np.float32)

def cebra_fit_beh(X_train, y_train, X_apply, cfg, seed, val=None, model_path=None):
    """Come cebra_fit (§2.3), ma con le etichette: CEBRA-Behavior. val = {nome: (X, y)}: insiemi su cui misurare,
    alla fine, la funzione di costo con la stessa scelta dei positivi. Gira nel processo separato."""
    import time                                             # per misurare la durata
    import numpy as np                                      # array
    import torch                                            # la libreria delle reti neurali usata da CEBRA
    import cebra                                            # CEBRA
    import cebra.distributions.base as cdb                  # per la correzione del seme (§2.7)
    torch.set_num_threads(1)                                # i calcoli pesanti sono sulla GPU
    if not hasattr(cdb.HasGenerator, "orig_init"):          # la stessa correzione del seme di cebra_fit (§2.7), codice identico
        cdb.HasGenerator.orig_init = cdb.HasGenerator.__init__
        def seeded_init(self, device, seed):
            cdb.HasGenerator.orig_init(self, device=device, seed=seed)
            if cdb.HasGenerator.fixed_seed is not None:
                self._seed = cdb.HasGenerator.fixed_seed * 1000 + cdb.HasGenerator.counter
                cdb.HasGenerator.counter += 1
                self._generator = torch.Generator(device=self.device).manual_seed(self._seed)
        cdb.HasGenerator.__init__ = seeded_init
    cdb.HasGenerator.fixed_seed = seed                      # sempre con la correzione
    cdb.HasGenerator.counter = 0
    torch.manual_seed(seed); np.random.seed(seed)
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = cebra.CEBRA(**cfg, device=device, verbose=False)
    t0 = time.time()
    model.fit(X_train, y_train)                             # con un'etichetta continua: CEBRA-Behavior (positivi scelti con "time_delta")
    seconds = time.time() - t0
    if model_path is not None:
        model.save(str(model_path))
    extra = {name: float(cebra.sklearn.metrics.infonce_loss(model, Xv, yv, num_batches=200))  # funzione di costo su dati dati, 200 gruppi
             for name, (Xv, yv) in (val or {}).items()}
    return {"maps": [model.transform(x) for x in X_apply], "loss": np.asarray(model.state_dict_["loss"]),
            "seconds": seconds, "device": device, "extra": extra}

print(f"Etichette: {Y_KIN.shape[1]} numeri per bin; deviazione standard prima dello z-score: "
      f"posizioni da {Y_KIN[:, :7].std(0).min():.2f} a {Y_KIN[:, :7].std(0).max():.2f}, "
      f"velocità da {Y_KIN[:, 7:].std(0).min():.2f} a {Y_KIN[:, 7:].std(0).max():.2f}")

In [ ]:
# dal §6.3 del 04 (senza UMAP e senza le misure del §6.3 che non servono qui): CEBRA-Behavior dalla cache e PLS
N_ITER_BEH = 1000
CEBRA_B = dict(CEBRA_STD, max_iterations=N_ITER_BEH)        # CEBRA-Behavior con il numero di passi del §6.2
YZ_F = [label_z(np.unique(trial[TRAIN_BINS[k]])) for k in range(5)]  # etichette z-scorate per blocco (statistiche dei trial di training)
res6 = run_cebra_cached([dict(X_train=X25[k][TRAIN_BINS[k]].astype(np.float32), y_train=YZ_F[k][TRAIN_BINS[k]],
                              X_apply=[X25[k][trial == t].astype(np.float32) for t in TRIALS],
                              cfg=dict(CEBRA_B, output_dimension=od), seed=sd) for od, sd, k in COMBOS], fn=cebra_fit_beh)
MAPS6 = {c: np.concatenate(r["maps"]) for c, r in zip(COMBOS, res6)}
for od in OUT_DIMS:
    ACC[f"CEBRA-Behavior, {od} dimensioni"] = [decode_map(fold_inp(lambda k: MAPS6[(od, sd, k)]), emb_none) for sd in SEEDS]

PLS_F = {nc: [PLSRegression(n_components=nc).fit(X25[k][TRAIN_BINS[k]], YZ_F[k][TRAIN_BINS[k]]) for k in range(5)]
         for nc in (2, 3)}                                  # PLS sugli stessi bin di training e con le stesse etichette di CEBRA

In [ ]:
# REPLICA: solo le definizioni della cella tolta 75
ITER_BEH = (1, 500, 1000, 2000, 5000, 10_000)

jobs_it, keys_it = [], []

N_ITER_BEH = 0

In [ ]:
# REPLICA: solo le definizioni della cella tolta 77
def decode_label(inp, embed, lab):                          # come decode_map, ma con un'altra etichetta da riconoscere
    b, folds = inp
    pred = np.empty(len(b), dtype=object)
    for Xtr, Xte, trn, tst in folds:
        Etr, Ete = embed(Xtr, Xte)
        pred[tst] = KNeighborsClassifier(n_neighbors=K_NN).fit(Etr, lab[b[trn]]).predict(Ete)
    return balanced_accuracy_score(lab[b], pred) * 100

def kin_r2(inp, embed, cols):                               # R² medio con cui un regressore a 10 vicini ricava alcune colonne della cinematica
    b, folds = inp
    pred = np.empty((len(b), len(cols)))
    for Xtr, Xte, trn, tst in folds:
        Etr, Ete = embed(Xtr, Xte)
        pred[tst] = KNeighborsRegressor(n_neighbors=K_NN).fit(Etr, Y_KIN[b[trn]][:, cols]).predict(Ete)
    return r2_score(Y_KIN[b][:, cols], pred) * 100

def umap_fold(d, sd, k):                                    # UMAP del blocco k come in emb_umap (§1.2): (punti di training, punti di test)
    Xtr, Xte, trn, tst = INP_PC[1][k]
    u = UMAP(n_neighbors=K_NN, n_components=d, min_dist=0.1, random_state=sd, n_jobs=1).fit(Xtr)
    return u.embedding_, u.transform(Xte)

def umap_inp(d, sd):                                        # l'input del metro comune con le mappe UMAP già stimate
    return (bins, [(*UMAP_F[(d, sd, k)], trn, tst) for k, (trn, tst) in enumerate(FOLDS)])

ROWS_6 = [("PCA, d = 2", 2, "no"), ("UMAP, d = 2", 2, "no"), ("CEBRA-Time, 3 dimensioni", 2, "no"),
          ("PLS, 2 componenti", 2, "sì"), ("CEBRA-Behavior, 3 dimensioni", 2, "sì"),
          ("PCA, d = 3", 3, "no"), ("UMAP, d = 3", 3, "no"), ("CEBRA-Time, 4 dimensioni", 3, "no"),
          ("PLS, 3 componenti", 3, "sì"), ("CEBRA-Behavior, 4 dimensioni", 3, "sì"),
          ("tetto, 25 dimensioni", 25, "no"), ("tetto alla pari: media sulla finestra (25 numeri)", 25, "no"),
          ("l'orologio da solo (1 numero)", 1, "(il tempo)")]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 79
def shuffled_labels(k, how, seed):                          # le etichette dei trial di training del blocco k, mescolate
    rng_s = np.random.default_rng(seed)
    tb = TRAIN_BINS[k]
    y_ = YZ_F[k][tb]
    if how == "tempo":                                      # ogni bin riceve l'etichetta di un bin a caso
        return y_[rng_s.permutation(len(tb))]
    trs = list(np.unique(trial[tb]))                        # i trial di training
    while True:                                             # un rimescolamento in cui nessun trial tiene la propria cinematica
        perm = rng_s.permutation(trs)
        if not np.any(perm == np.array(trs)):
            break
    out = []
    for t, t_other in zip(trs, perm):                       # ogni trial riceve la cinematica di un altro, allineata all'inizio
        n_t, y_o = (trial == t).sum(), YZ_F[k][trial == t_other]
        out.append(y_o[:n_t] if len(y_o) >= n_t else np.r_[y_o, np.repeat(y_o[-1:], n_t - len(y_o), axis=0)])
    return np.concatenate(out).astype(np.float32)

CONTROLS = ("tempo", "trial")

In [ ]:
# REPLICA: solo le definizioni della cella tolta 82
from sklearn.linear_model import Ridge

rows_k = {}

def kin_pred_r2(make_model):                                # R² di ogni grado di libertà, con un modello stimato sui punti di training di ogni blocco
    pred = np.empty((len(bins), Y_KIN.shape[1]))
    for Xtr, Xte, trn, tst in INP_PC[1]:
        pred[tst] = make_model().fit(Xtr, Y_KIN[bins[trn]]).predict(Xte)
    return r2_score(Y_KIN[bins], pred, multioutput="raw_values")

In [ ]:
# REPLICA: solo le definizioni della cella tolta 85
SEEDS3 = (0, 1, 2)

def variant_jobs(cfg, X_full=None, train_bins=None, labels=None):  # gli allenamenti di una variante: 3 semi × 5 blocchi
    jobs = []
    for sd in SEEDS3:
        for k in range(5):
            Xk = (X25[k] if X_full is None else X_full[k]).astype(np.float32)  # input di tutta la sessione per il blocco k
            tb = TRAIN_BINS[k] if train_bins is None else train_bins[k]       # i bin su cui si allena
            j = dict(X_train=Xk[tb], X_apply=[Xk[trial == t] for t in TRIALS], cfg=cfg, seed=sd)
            if labels is not None:
                j["y_train"] = labels[k][tb]                # le etichette (solo per CEBRA-Behavior, §7c)
            jobs.append(j)
    return jobs

def run_variants(variants, fn=None):                        # tutte le varianti in un'unica coda; restituisce le 3 accuratezze di ognuna
    names, jobs = [], []
    for name, v in variants.items():
        js = variant_jobs(**v)
        jobs += js; names += [name] * len(js)
    res = run_cebra_cached(jobs, fn=fn)
    out, maps_out = {}, {}
    for name in variants:
        rs = [r for n_, r in zip(names, res) if n_ == name] # i 15 risultati della variante: seme esterno, blocco interno
        maps = {(sd, k): np.concatenate(rs[i * 5 + k]["maps"]) for i, sd in enumerate(SEEDS3) for k in range(5)}
        out[name] = [decode_map(fold_inp(lambda k, sd=sd: maps[(sd, k)]), emb_none) for sd in SEEDS3]
        maps_out[name] = maps
    return out, maps_out

MAIN = "principale (3 dimensioni, τ = 1, offset10, 1000 passi, coseno)"

def sens_table(acc, ref_name):                              # tabella: media (min-max) e differenza dalla principale, seme per seme
    ref = np.array(acc[ref_name])
    return pd.DataFrame({name: {"fase riconosciuta (%), 3 semi": mean_range(v),
                                "differenza dalla principale (punti), media": f"{np.mean(np.array(v) - ref):+.1f}",
                                "differenza, seme per seme": " ".join(f"{d_:+.1f}" for d_ in np.array(v) - ref)}
                         for name, v in acc.items()}).T

In [ ]:
# REPLICA: solo le definizioni della cella tolta 87
HI_ACC = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 90
import contextlib, logging

from elephant.gpfa import gpfa_core

import elephant.utils

def make_seqs(segs):                                        # i dati nel formato di elephant: un pezzo (inizio, fine) per riga
    return np.array([(b_ - a_, Y_SQRT[a_:b_].T) for a_, b_ in segs], dtype=[("T", int), ("y", "O")])

def fit_gpfa_fold(k):                                       # GPFA a 25 fattori sulle finestre da 1 s dei trial di training del blocco k (03, §7.2)
    train_tr = np.unique(trial[TRAIN_BINS[k]])
    np.random.seed(0)                                       # punto di partenza fisso
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        params, _ = gpfa_core.fit(make_seqs([(w["start"], w["stop"]) for w in windows if w["trial"] in train_tr]),
                                  x_dim=K_PCA, bin_width=20.0, em_max_iters=500, tau_init=100.0)
    return params

SEG, KEEP = 100, 50

def gpfa_continuous(params):                                # i 25 fattori ordinati di ogni bin della sessione, a pezzi sovrapposti dentro ogni trial
    segs, owner = [], []
    for t in TRIALS:
        a0, a1 = first_last[t][0], first_last[t][1] + 1     # inizio e fine del trial
        starts = list(range(a0, a1 - SEG + 1, KEEP))
        if starts[-1] != a1 - SEG:
            starts.append(a1 - SEG)                         # l'ultimo pezzo finisce esattamente alla fine del trial
        centers = np.array(starts) + SEG / 2
        for x in range(a0, a1):                             # ogni bin viene dal pezzo in cui è più centrale
            owner.append((x, len(segs) + int(np.argmin(np.abs(centers - x)))))
        segs += [(a_, a_ + SEG) for a_ in starts]
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        lat, _ = gpfa_core.exact_inference_with_ll(make_seqs(segs), params, get_ll=False)
        _, orth = gpfa_core.orthonormalize(params, lat)     # fattori ruotati e ordinati per importanza
    out = np.empty((n_bins, K_PCA))
    for x, i in owner:
        out[x] = orth["latent_variable_orth"][i][:, x - segs[i][0]]
    return out

rows_b = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 92
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

HAND = [10, 11, 12, 13]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 96
def row(name, lab, a2=None, a3=None, a7=None):              # una riga della tabella finale
    return {"metodo": name, "etichette": lab, "2 coordinate libere": fmt(a2) if a2 is not None else "",
            "3 coordinate libere": fmt(a3) if a3 is not None else "", "7 coordinate libere": fmt(a7) if a7 is not None else ""}

def tstat(name, col):                                       # una misura delle traiettorie (§4.2): media (min-max) tra i semi
    v = np.array([r_[col] for r_ in TRAJ[name]])
    return f"{v.mean():.3f}" if len(v) == 1 else f"{v.mean():.3f} ({v.min():.3f}-{v.max():.3f})"

In [ ]:
# REPLICA: solo le definizioni della cella tolta 97
METH = ["PCA", "Isomap", "UMAP", "GPFA (primi fattori)", "CEBRA-Time (3, 4 e 8 dimensioni)",
        "CEBRA-Behavior, posizione + velocità", "PLS, posizione + velocità"]

COLS8 = {"PCA": "C1", "Isomap": "C0", "UMAP": "C2", "GPFA (primi fattori)": "C4", "CEBRA-Time (3, 4 e 8 dimensioni)": "C3",
         "CEBRA-Behavior, posizione + velocità": "#f4a6a6", "PLS, posizione + velocità": "0.6"}

w = 0.11

In [ ]:
# REPLICA: solo le definizioni della cella tolta 104
SEG_LEN, SEG_OVL = 30, 10

def segment_starts(a0, a1):                                 # dove cominciano i pezzi di un trial che va dal bin a0 al bin a1 (escluso)
    st = list(range(a0, a1 - SEG_LEN + 1, SEG_STEP))        # un pezzo ogni 20 bin...
    if st[-1] != a1 - SEG_LEN:
        st.append(a1 - SEG_LEN)                             # ...più un ultimo pezzo che finisce esattamente con il trial
    return np.array(st)

def cut_segments(X, t):                                     # i pezzi del trial t di un segnale X (bin × colonne): (pezzi, 30 bin, colonne)
    return np.stack([X[a:a + SEG_LEN] for a in SEG_START[t]])

def segment_weights(n_seg):                                 # i pesi dei 30 bin di ognuno degli n_seg pezzi di un trial
    W = np.ones((n_seg, SEG_LEN))                           # 1 dove il pezzo è da solo
    W[1:, :SEG_OVL] = RAMP_IN                               # tutti i pezzi tranne il primo del trial cominciano con la rampa in salita
    W[:-1, -SEG_OVL:] = RAMP_OUT                            # tutti tranne l'ultimo finiscono con la rampa in discesa
    return W

def stitch(Y_seg, t):                                       # ricuce i pezzi del trial t (pezzi × 30 bin × colonne) in un segnale continuo (bin del trial × colonne)
    a0, a1 = TRIAL_SPAN[t]
    num, den = np.zeros((a1 - a0, Y_seg.shape[2])), np.zeros((a1 - a0, 1))
    for a, y, w in zip(SEG_START[t] - a0, Y_seg, segment_weights(len(Y_seg))):
        num[a:a + SEG_LEN] += w[:, None] * y                # somma dei valori pesati...
        den[a:a + SEG_LEN] += w[:, None]                    # ...e dei pesi
    return num / den

In [ ]:
# REPLICA: solo le definizioni della cella tolta 106
VAL_LF = []

def n_seg(trs):                                             # numero di pezzi di un gruppo di trial
    return sum(len(SEGS[t]) for t in trs)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 108
def describe(ch):                                           # frequenze e quota di spike di un gruppo di canali
    return {"canali": len(ch), "frequenza mediana (Hz)": np.median(rate_hz[ch]),
            "da (Hz)": rate_hz[ch].min(), "a (Hz)": rate_hz[ch].max(),
            "spike sul totale (%)": COUNTS[:, ch].sum() / COUNTS.sum() * 100}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 111
show_n = 300

SEG_COL = ("k", "0.55")

In [ ]:
# REPLICA: solo le definizioni della cella tolta 114
SIGMA_L, RHO_L, BETA_L = 10.0, 28.0, 8 / 3

DT_BIN, N_SUB = 0.015, 10

T_LZ, N_NEU = 10_000, 30

def lorenz_step(v, dt):                                     # un passo di Eulero: stato nuovo = stato + dt × (velocità data dalle regole di Lorenz)
    x, y, z = v
    return v + dt * np.array([SIGMA_L * (y - x), x * (RHO_L - z) - y, x * y - BETA_L * z])

LZ_PARTS = {"impara": (0, 8_000), "validazione": (8_000, 9_000), "test": (9_000, 10_000)}

def nll_sum(k, lam):                                        # errore di Poisson totale, senza log k! (è uguale per tutti i modelli e si cancella nei confronti)
    return np.sum(lam - k * np.log(lam))

In [ ]:
# REPLICA: solo le definizioni della cella tolta 117
def sigmoid(x):                                             # la funzione a "S" tra 0 e 1, usata dalle porte
    return 1 / (1 + np.exp(-x))

def gru_step(x, h, w, name, clip):                          # un passo della GRU di lfads-torch (ClippedGRUCell): ingresso x, stato h, pesi w[name + ...]
    H = h.shape[1]                                          # dimensione dello stato
    W_ih, b_ih = w[name + "weight_ih"], w[name + "bias_ih"] # pesi dell'ingresso (per le 3 porte, una sopra l'altra) e loro livelli
    W_hh, b_hh = w[name + "weight_hh"], w[name + "bias_hh"] # pesi dello stato precedente, e loro livelli
    x_z, x_r, x_n = np.split(x @ W_ih.T + b_ih, 3, axis=1)  # il contributo dell'ingresso alle porte z, r e alla proposta n (in quest'ordine)
    h_z, h_r = np.split(h @ W_hh[:2 * H].T + b_hh[:2 * H], 2, axis=1)  # il contributo dello stato precedente alle porte z e r
    z = sigmoid(x_z + h_z)                                  # porta di aggiornamento: quanto tenere dello stato vecchio (da 0 a 1)
    r = sigmoid(x_r + h_r)                                  # porta di reset: quanto dello stato vecchio usare per fare la proposta
    n = np.tanh(x_n + (r * h) @ W_hh[2 * H:].T + b_hh[2 * H:])  # la proposta di stato nuovo (r moltiplica lo stato PRIMA dei pesi)
    return np.clip(z * h + (1 - z) * n, -clip, clip)

def lfads_forward_np(data, w, hp):
    """LFADS a mano: dai conteggi (pezzi × bin × neuroni) a condizione iniziale, stati, fattori e log-frequenze.
    Versione di valutazione: si usa la media della condizione iniziale e nessun dropout."""
    B, T, _ = data.shape                                    # pezzi, bin per pezzo, neuroni
    clip = hp["cell_clip"]
    h_f = np.tile(w["encoder.ic_enc_h0"][0], (B, 1))        # 1. encoder: stato di partenza della GRU che legge in avanti (un parametro imparato)...
    h_b = np.tile(w["encoder.ic_enc_h0"][1], (B, 1))        # ...e di quella che legge all'indietro
    for t in range(T):                                      # dal primo bin all'ultimo
        h_f = gru_step(data[:, t], h_f, w, "encoder.ic_enc.fwd_gru.cell.", clip)
    for t in reversed(range(T)):                            # dall'ultimo bin al primo
        h_b = gru_step(data[:, t], h_b, w, "encoder.ic_enc.bwd_gru.cell.", clip)
    summary = np.concatenate([h_f, h_b], axis=1)            # il riassunto del pezzo: i due stati finali, uno accanto all'altro
    ic_params = summary @ w["encoder.ic_linear.weight"].T + w["encoder.ic_linear.bias"]  # 2. condizione iniziale: una trasformazione lineare...
    ic_mean, ic_logvar = np.split(ic_params, 2, axis=1)     # ...che dà la media e il logaritmo della varianza
    ic_std = np.sqrt(np.exp(ic_logvar) + hp["ic_post_var_min"])  # l'incertezza (deviazione standard), mai sotto un minimo
    g = ic_mean @ w["decoder.ic_to_g0.weight"].T + w["decoder.ic_to_g0.bias"]  # 3. generatore: lo stato di partenza g₀, dalla media della condizione iniziale
    W_fac = w["decoder.rnn.cell.fac_linear.weight"]         # 4. le direzioni dei fattori nello stato del generatore...
    W_fac = W_fac / np.linalg.norm(W_fac, axis=1, keepdims=True)  # ...ognuna normalizzata a lunghezza 1 (KernelNormalizedLinear)
    no_input = np.zeros((B, 0))                             # il generatore non ha ingressi: un ingresso "vuoto" (0 colonne)
    gen, fac = [], []
    for t in range(T):                                      # bin dopo bin, solo le regole del generatore
        g = gru_step(no_input, g, w, "decoder.rnn.cell.gen_cell.", clip)  # senza ingresso, x @ W_ih vale 0: restano i livelli b_ih
        gen.append(g)
        fac.append(g @ W_fac.T)                             # i fattori del bin
    gen, fac = np.stack(gen, 1), np.stack(fac, 1)           # (pezzi, bin, 64) e (pezzi, bin, 10)
    log_rates = fac @ w["readout.0.weight"].T + w["readout.0.bias"]  # 5. log-frequenze: combinazione lineare dei fattori più un livello per neurone
    return {"ic_mean": ic_mean, "ic_std": ic_std, "gen_states": gen, "factors": fac, "log_rates": log_rates}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 119
from joblib.externals.loky import ProcessPoolExecutor

def make_lfads(hp):
    """Un LFADS della libreria lfads-torch costruito dai parametri hp, senza controller. Si chiama nel processo separato."""
    import numpy as np                                      # numpy PRIMA di torch: altrimenti Python si chiude (§10.3)
    import torch
    from torch import nn
    from lfads_torch.model import LFADS                     # il modello della libreria
    from lfads_torch.modules import augmentations           # il coordinated dropout
    from lfads_torch.modules.priors import MultivariateNormal, Null  # la distribuzione a priori della condizione iniziale; "nessuna" per il controller
    from lfads_torch.modules.recons import Poisson          # il modello del rumore: Poisson
    from lfads_torch.modules.readin_readout import FanInLinear  # l'uscita lineare dai fattori alle log-frequenze
    cd = augmentations.CoordinatedDropout(cd_rate=hp["cd_rate"], cd_pass_rate=0.0, ic_enc_seq_len=0)
    return LFADS(encod_data_dim=hp["n_in"], encod_seq_len=hp["seg_len"], recon_seq_len=hp["seg_len"],  # forma dei dati
                 ext_input_dim=0, ic_enc_seq_len=0,         # nessun ingresso esterno; l'encoder legge tutto il pezzo
                 ic_enc_dim=hp["enc_dim"], ci_enc_dim=0, ci_lag=1, con_dim=0, co_dim=0,  # encoder; controller spento (dimensioni 0)
                 ic_dim=hp["ic_dim"], gen_dim=hp["gen_dim"], fac_dim=hp["fac_dim"], dropout_rate=hp["dropout"],
                 reconstruction=nn.ModuleList([Poisson()]), variational=True, co_prior=Null(),
                 ic_prior=MultivariateNormal(mean=0.0, variance=hp["ic_prior_var"], shape=hp["ic_dim"]),
                 ic_post_var_min=hp["ic_post_var_min"], cell_clip=hp["cell_clip"],
                 train_aug_stack=augmentations.AugmentationStack([cd], batch_order=[0], loss_order=[0]),  # coordinated dropout solo in allenamento
                 infer_aug_stack=augmentations.AugmentationStack(),
                 readin=nn.ModuleList([nn.Identity()]), readout=nn.ModuleList([FanInLinear(hp["fac_dim"], hp["n_out"])]),
                 loss_scale=hp["loss_scale"], recon_reduce_mean=True,  # errore di ricostruzione come media su pezzi, bin e neuroni
                 lr_scheduler=False, lr_init=hp["lr"], lr_stop=1e-5, lr_decay=0.95, lr_patience=6,
                 lr_adam_beta1=0.9, lr_adam_beta2=0.999, lr_adam_epsilon=1e-8, weight_decay=0.0,
                 l2_start_epoch=0, l2_increase_epoch=hp["ramp_epochs"], l2_ic_enc_scale=hp["l2_ic_enc_scale"],
                 l2_ci_enc_scale=0.0, l2_gen_scale=hp["l2_gen_scale"], l2_con_scale=0.0,
                 kl_start_epoch=0, kl_increase_epoch=hp["ramp_epochs"], kl_ic_scale=hp["kl_ic_scale"], kl_co_scale=0.0)

def lfads_library_check(hp, seed, data):
    """Nel processo separato: un modello della libreria con i pesi iniziali (seme `seed`), le sue uscite e il suo costo su `data`."""
    import numpy as np                                      # numpy PRIMA di torch
    import torch
    from lfads_torch.tuples import SessionBatch             # il formato dei dati della libreria
    from lfads_torch.modules.l2 import compute_l2_penalty   # la penalità L2 della libreria
    torch.set_num_threads(1)                                # un solo thread di CPU (§10b: di più non serve)
    torch.manual_seed(seed)                                 # seme dei pesi iniziali
    model = make_lfads(hp)
    model.eval()                                            # modalità di valutazione: niente dropout
    model.log = model.log_dict = lambda *a, **k: None       # fuori dall'allenatore di Lightning non c'è dove scrivere le misure
    X = torch.tensor(data, dtype=torch.float32)
    batch = SessionBatch(encod_data=X, recon_data=X, ext_input=torch.zeros(len(X), X.shape[1], 0),
                         truth=torch.zeros(0), sv_mask=torch.ones_like(X))  # ingresso = dati da ricostruire; nessun ingresso esterno
    with torch.no_grad():                                   # niente derivate: solo il calcolo in avanti
        out = model({0: batch}, sample_posteriors=False, output_means=False)[0]  # tutte le uscite, con la media della condizione iniziale
        recon = model.recon[0].compute_loss(X, out.output_params).mean()  # errore di ricostruzione (Poisson) della libreria
        kl = model.ic_prior(out.ic_mean, out.ic_std)        # termine KL della libreria
        l2 = compute_l2_penalty(model, model.hparams)       # penalità L2 della libreria
        model.hparams.variational = False                   # per il controllo: il costo con la media, senza estrazioni a caso
        total = model._shared_step({0: (batch, ())}, 0, "valid")  # il costo totale, calcolato dalla funzione che la libreria usa in allenamento
    w = {k: v.detach().numpy().astype(np.float64) for k, v in model.state_dict().items()}  # tutti i pesi, come array numpy
    return {"weights": w, "ic_mean": out.ic_mean.numpy(), "ic_std": out.ic_std.numpy(), "gen_states": out.gen_states.numpy(),
            "factors": out.factors.numpy(), "log_rates": out.output_params[..., 0].numpy(),
            "recon": float(recon), "kl": float(kl), "l2": float(l2), "total": float(total)}

OUT_NAMES = {"ic_mean": "condizione iniziale: media", "ic_std": "condizione iniziale: incertezza",
             "gen_states": "stati del generatore", "factors": "fattori", "log_rates": "log-frequenze"}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 121
from scipy.special import gammaln

def stirling(k):                                            # l'approssimazione di log k! usata da PyTorch (full=True): solo per k > 1, altrimenti 0
    k_safe = np.where(k > 1, k, 1.0)                        # per k ≤ 1 un valore qualunque, che poi non si usa (evita log 0)
    return np.where(k > 1, k_safe * np.log(k_safe) - k_safe + 0.5 * np.log(2 * np.pi * k_safe), 0.0)

def poisson_nll(k, log_rate, exact=True):                   # errore di Poisson di ogni bin e neurone: λ − k log λ + log k!
    return np.exp(log_rate) - k * log_rate + (gammaln(k + 1) if exact else stirling(k))

def kl_normal(mu, sd, prior_mean, prior_var):               # KL tra la condizione iniziale (μ, σ) e quella a priori, sommato sulle dimensioni, medio sui pezzi
    sp = np.sqrt(prior_var)
    return np.mean(np.sum(np.log(sp / sd) + (sd ** 2 + (mu - prior_mean) ** 2) / (2 * prior_var) - 0.5, axis=1))

def l2_gen(w, hp):                                          # penalità L2 della libreria: ½ ‖pesi ricorrenti‖², per il suo peso, diviso il numero di pesi
    terms = [(w["decoder.rnn.cell.gen_cell.weight_hh"], hp["l2_gen_scale"]),
             (w["encoder.ic_enc.fwd_gru.cell.weight_hh"], hp["l2_ic_enc_scale"]),
             (w["encoder.ic_enc.bwd_gru.cell.weight_hh"], hp["l2_ic_enc_scale"])]
    used = [(W_, s_) for W_, s_ in terms if s_ > 0]         # come la libreria: solo i pesi con una penalità diversa da 0
    return sum(s_ * 0.5 * np.sum(W_ ** 2) for W_, s_ in used) / (sum(W_.size for W_, _ in used) + 1e-8)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 124
import copy

CLIP_LF = 200 / 1e4

def lfads_fit(hp, seed, train, val, apply, watch=None, max_epochs=1000, patience=200, batch_size=64, clip=CLIP_LF,
              device="cpu", model_path=None, rate_cols=None):
    """Allena un LFADS della libreria (make_lfads) sui pezzi `train` (pezzi × bin × neuroni) e sceglie la versione con
    l'errore più basso sui pezzi `val`. Restituisce, per ogni array di `apply`, fattori e frequenze (media della
    condizione iniziale, senza dropout; delle frequenze solo le colonne rate_cols, se indicate), la storia
    dell'allenamento e i pesi. watch = {nome: (pezzi, frequenze vere o None[, colonne nascoste all'encoder])}: altre
    misure a ogni epoca (errore, bit per spike, R² con la verità). Gira nel processo separato."""
    import copy, time                                       # copia della versione migliore; durata
    import numpy as np                                      # numpy PRIMA di torch (§10.3)
    import torch
    from lfads_torch.tuples import SessionBatch             # il formato dei dati della libreria
    torch.set_num_threads(1)                                # un thread: i calcoli sono piccoli (§10.8)
    torch.manual_seed(seed)                                 # pesi iniziali, coordinated dropout, dropout, estrazioni della condizione iniziale
    rng = np.random.default_rng(seed)                       # l'ordine dei pezzi in ogni epoca
    model = make_lfads(hp)
    model.__class__ = type("LFADSconEpoca", (type(model),), {"current_epoch": 0})  # un numero d'epoca che possiamo scrivere noi (per le rampe)
    model = model.to(device)
    model.log = model.log_dict = lambda *a, **k: None       # fuori da Lightning non c'è dove scrivere le misure
    opt = model.configure_optimizers()                      # AdamW con il passo di apprendimento di hp, come la libreria
    n_in = hp["n_in"]                                       # i neuroni che l'encoder vede (i primi n_in; al §12 non saranno tutti)

    def as_batch(X):                                        # pezzi numpy -> il formato della libreria
        Xt = torch.tensor(X, dtype=torch.float32, device=device)
        return SessionBatch(encod_data=Xt[..., :n_in], recon_data=Xt,  # l'encoder vede i primi n_in neuroni, si ricostruiscono tutti
                            ext_input=torch.zeros(len(Xt), Xt.shape[1], 0, device=device),
                            truth=torch.zeros(0), sv_mask=torch.ones_like(Xt))

    def evaluate(X, chunk=256):                             # fattori, log-frequenze ed errore medio, con la media della condizione iniziale e senza dropout
        model.eval()
        fac, logr, err = [], [], []
        with torch.no_grad():
            for i in range(0, len(X), chunk):
                sb = as_batch(X[i:i + chunk])
                out = model({0: sb}, sample_posteriors=False, output_means=False)[0]
                fac.append(out.factors.cpu().numpy())
                logr.append(out.output_params[..., 0].cpu().numpy())
                err.append(model.recon[0].compute_loss(sb.recon_data, out.output_params).sum().item())  # errore di Poisson della libreria (Stirling)
        return np.concatenate(fac), np.concatenate(logr), sum(err) / X.size

    def bits(logr, X, cols):                                # bit per spike sulle colonne cols: guadagno sulla frequenza media di ogni canale in questi dati (§9.4)
        k, lr = X[..., cols].reshape(-1, len(cols)), logr[..., cols].reshape(-1, len(cols))
        lam0 = np.maximum(k.mean(0), 1e-9)                  # la frequenza media di ogni canale (mai esattamente 0)
        nll_model = np.sum(np.exp(lr) - k * lr)             # errore di Poisson del modello (senza log k!, che si cancella)
        nll_null = np.sum(lam0 - k * np.log(lam0))          # errore di chi conosce solo la media
        return float((nll_null - nll_model) / k.sum() / np.log(2))

    def r2_rates(logr, truth):                              # R² tra le frequenze del modello e quelle vere, medio sui neuroni
        p_, t_ = np.exp(logr).reshape(-1, truth.shape[-1]), truth.reshape(-1, truth.shape[-1])
        return float(np.mean(1 - ((t_ - p_) ** 2).sum(0) / ((t_ - t_.mean(0)) ** 2).sum(0)))

    hist, best_err, best_state, best_epoch = [], np.inf, None, 0
    t0, steps = time.time(), 0
    for epoch in range(1, max_epochs + 1):
        model.current_epoch = epoch - 1                     # la libreria conta le epoche da 0
        model.train()                                       # modalità di allenamento: dropout acceso
        order = rng.permutation(len(train))                 # i pezzi in un ordine nuovo a ogni epoca
        costs = []
        for i in range(0, len(train), batch_size):          # a gruppi da 64 pezzi
            loss = model._shared_step({0: (as_batch(train[order[i:i + batch_size]]), ())}, i, "train")  # il costo della libreria (con CD ed estrazione a caso)
            opt.zero_grad()                                 # azzera le derivate del passo precedente
            loss.backward()                                 # le derivate del costo rispetto a tutti i pesi
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)  # taglio delle derivate troppo lunghe
            opt.step()                                      # un passo di AdamW
            costs.append(loss.item()); steps += 1
        val_err = evaluate(val)[2]                          # errore sui pezzi di validazione
        row = {"epoca": epoch, "costo di training": float(np.mean(costs)), "errore, validazione": val_err}
        for name, (Xw, truth, *hid) in (watch or {}).items():  # misure in più (per i dati sintetici, anche la distanza dalla verità)
            _, lw, ew = evaluate(Xw)
            row[f"errore, {name}"] = ew
            if truth is not None:
                row[f"R² frequenze, {name}"] = r2_rates(lw, truth)
            if hid:                                         # canali nascosti all'encoder: bit per spike separati per visibili e nascosti
                vis = np.setdiff1d(np.arange(Xw.shape[-1]), hid[0])
                row[f"bit per spike visibili, {name}"] = bits(lw, Xw, vis)
                row[f"bit per spike nascosti, {name}"] = bits(lw, Xw, np.asarray(hid[0]))
            else:
                row[f"bit per spike, {name}"] = bits(lw, Xw, np.arange(Xw.shape[-1]))
        hist.append(row)
        if epoch >= hp["ramp_epochs"] and val_err < best_err:  # dopo le rampe: teniamo la versione migliore sulla validazione
            best_err, best_state, best_epoch = val_err, copy.deepcopy(model.state_dict()), epoch
        if best_epoch > 0 and epoch - best_epoch >= patience:  # 200 epoche senza miglioramenti: ci si ferma
            break
    if best_state is not None:
        model.load_state_dict(best_state)                   # la versione migliore
    seconds = time.time() - t0
    if model_path is not None:
        torch.save(model.state_dict(), model_path)          # il modello, su disco
    outs = [evaluate(X)[:2] for X in apply]                 # fattori e log-frequenze di ogni array da proiettare
    keep = slice(None) if rate_cols is None else np.asarray(rate_cols)  # quali frequenze tenere (per non riempire il disco)
    return {"factors": [f_.astype(np.float32) for f_, _ in outs], "rates": [np.exp(l_[..., keep]).astype(np.float32) for _, l_ in outs],
            "history": hist, "best_epoch": best_epoch, "epochs": len(hist), "steps": steps, "seconds": seconds,
            "weights": {k: v.detach().cpu().numpy() for k, v in model.state_dict().items()}}

def lf_key(job):                                            # impronta di un allenamento LFADS: cambia se cambia qualunque cosa che lo determina
    h = hashlib.sha1()
    for f in (lfads_fit, make_lfads):
        h.update(inspect.getsource(f).encode())             # il codice
    h.update(json.dumps({k: job.get(k) for k in ("hp", "seed", "max_epochs", "patience", "batch_size", "clip", "device", "rate_cols")},
                        sort_keys=True, default=str).encode())  # parametri, seme, regole dell'allenamento, frequenze tenute
    arrays = [job["train"], job["val"], *job["apply"]]      # i dati...
    for name, (Xw, truth, *hid) in sorted((job.get("watch") or {}).items()):
        h.update(name.encode()); arrays += [Xw] + ([] if truth is None else [truth]) + [np.asarray(c_) for c_ in hid]  # ...e quelli delle misure in più
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

def run_lfads_cached(jobs):                                 # esegue gli allenamenti che non sono già su disco, 4 alla volta; poi rilegge tutto dal disco
    keys = [lf_key(j) for j in jobs]
    todo = [i for i, k in enumerate(keys) if RETRAIN or not (CACHE_LF / f"{k}.npz").exists()]
    futures = {i: POOL_LF.submit(lfads_fit, **dict(jobs[i], model_path=CACHE_LF / f"{keys[i]}.pt")) for i in todo}
    for i, fut in futures.items():                          # i risultati nuovi si salvano appena arrivano
        r = fut.result()
        np.savez(CACHE_LF / f"{keys[i]}.npz", history=json.dumps(r["history"]),
                 info=json.dumps({k: r[k] for k in ("best_epoch", "epochs", "steps", "seconds")}),
                 **{f"factors_{j}": f_ for j, f_ in enumerate(r["factors"])}, **{f"rates_{j}": x_ for j, x_ in enumerate(r["rates"])})
        np.savez(CACHE_LF / f"{keys[i]}_w.npz", **r["weights"])  # i pesi, anche come array numpy (per il calcolo a mano)
    out = []
    for k in keys:                                          # tutti i risultati, nell'ordine dei lavori, letti dal disco
        z, zw = np.load(CACHE_LF / f"{k}.npz"), np.load(CACHE_LF / f"{k}_w.npz")
        n_ = sum(f_.startswith("factors_") for f_ in z.files)
        out.append({"factors": [z[f"factors_{j}"] for j in range(n_)], "rates": [z[f"rates_{j}"] for j in range(n_)],
                    "history": pd.DataFrame(json.loads(str(z["history"]))).set_index("epoca"),
                    **json.loads(str(z["info"])), "weights": {f_: zw[f_] for f_ in zw.files}, "key": k})
    print(f"Allenamenti LFADS: {len(todo)} eseguiti ora, {len(jobs) - len(todo)} riletti dal disco")
    return out

In [ ]:
# REPLICA: solo le definizioni della cella tolta 126
def syn_job(hp, seed):                                      # un allenamento sui dati sintetici
    return dict(hp=hp, seed=seed, train=LZ_SEGS["impara"], val=LZ_SEGS["validazione"],
                apply=[LZ_SEGS[k] for k in LZ_PARTS], watch=LZ_WATCH)

SYN_RUNS = [("con CD", 0), ("con CD", 1), ("con CD", 2), ("senza CD", 0)]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 129
from sklearn.linear_model import PoissonRegressor

def stitch_at(Y_seg, starts, a0, a1):                       # la ricucitura del §9.2 per un tratto qualunque [a0, a1) con i pezzi che cominciano in `starts`
    num, den = np.zeros((a1 - a0, Y_seg.shape[2])), np.zeros((a1 - a0, 1))
    for a, y, w in zip(np.asarray(starts) - a0, Y_seg, segment_weights(len(Y_seg))):
        num[a:a + SEG_LEN] += w[:, None] * y
        den[a:a + SEG_LEN] += w[:, None]
    return num / den

def stitch_lz(per_part):                                    # ricuce le uscite delle tre parti (impara, validazione, test) e le mette in fila: 10 000 bin
    return np.concatenate([stitch_at(Y, LZ_START[k], *LZ_PARTS[k]) for k, Y in zip(LZ_PARTS, per_part)])

def gpfa_seqs(segs):                                        # pezzi di conteggi -> il formato di elephant, con la radice dei conteggi (come nel 03 e nel §7b)
    return np.array([(s_.shape[0], np.sqrt(s_.astype(float)).T) for s_ in segs], dtype=[("T", int), ("y", "O")])

def gpfa_fit_segments(segs, x_dim):                         # GPFA con x_dim fattori stimata sui pezzi dati
    np.random.seed(0)                                       # punto di partenza fisso
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        params, _ = gpfa_core.fit(gpfa_seqs(segs), x_dim=x_dim, bin_width=20.0, em_max_iters=500, tau_init=100.0)
    return params

def gpfa_sqrt_pred(segs, params):                           # la previsione μ = C x + d della GPFA per la radice dei conteggi, in ogni pezzo: (pezzi, bin, neuroni)
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        lat, _ = gpfa_core.exact_inference_with_ll(gpfa_seqs(segs), params, get_ll=False)
    return np.stack([(params["C"] @ x_ + params["d"][:, None]).T for x_ in lat["latent_variable"]])

def gpfa_infer_segments(segs, params):                      # i fattori ordinati (orthonormalize) di ogni pezzo: (pezzi, bin, fattori)
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        lat, _ = gpfa_core.exact_inference_with_ll(gpfa_seqs(segs), params, get_ll=False)
        _, orth = gpfa_core.orthonormalize(params, lat)
    return np.stack([x_.T for x_ in orth["latent_variable_orth"]])

def lorenz_r2(F):                                           # R² delle 3 variabili di Lorenz ricavate linearmente da F (stimato sul training, giudicato sul test)
    reg = LinearRegression().fit(F[FIT_LZ], LZ_Z[FIT_LZ])
    return r2_score(LZ_Z[TEST_LZ], reg.predict(F[TEST_LZ]), multioutput="raw_values")

def rate_r2(R):                                             # R² tra frequenze stimate e vere sul test, medio sui 30 neuroni
    return float(np.mean(r2_score(RATE_LZ[TEST_LZ], R[TEST_LZ], multioutput="raw_values")))

def rate_corr(R):                                           # correlazione tra frequenze stimate e vere sul test, media sui 30 neuroni
    return float(np.mean([np.corrcoef(RATE_LZ[TEST_LZ, n], R[TEST_LZ, n])[0, 1] for n in range(N_NEU)]))

def glm_rates(F):                                           # frequenze da una regressione di Poisson per neurone, dai fattori agli spike, stimata sul training
    sc = StandardScaler().fit(F[FIT_LZ])
    Fs = sc.transform(F)
    return np.column_stack([PoissonRegressor(alpha=1e-4, max_iter=1000).fit(Fs[FIT_LZ], SPK_LZ[FIT_LZ, n]).predict(Fs)
                            for n in range(N_NEU)])

def lz_row(F, R):                                           # una riga della tabella: stato (da F) e frequenze (R)
    return {**dict(zip("xyz", lorenz_r2(F))), "frequenze: R²": rate_r2(R), "frequenze: correlazione": rate_corr(R)}

def cell_fmt(vals):                                         # "media (minimo-massimo)" se ci sono più semi
    vals = np.array([x_ for x_ in vals if x_ is not None and not np.isnan(x_)])
    if len(vals) == 0:
        return ""
    return f"{vals.mean():.3f}" if len(vals) == 1 else f"{vals.mean():.3f} ({vals.min():.3f}-{vals.max():.3f})"

COLS_LZ = ["x", "y", "z", "frequenze: R²", "frequenze: correlazione"]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 132
def seam_gap(Fseg, starts):                                 # nelle zone comuni a due pezzi consecutivi: differenza media tra i loro fattori, in deviazioni standard
    sd_ = Fseg.reshape(-1, Fseg.shape[2]).std(0)
    gaps = []
    for i in range(len(starts) - 1):
        ov = starts[i] + SEG_LEN - starts[i + 1]            # quanti bin hanno in comune
        if ov > 0:
            gaps.append(np.abs(Fseg[i, -ov:] - Fseg[i + 1, :ov]) / sd_)
    return float(np.mean(np.concatenate(gaps)))

def stitch_test_w(Y_seg, W):                                # ricucitura dei pezzi di test con pesi dati
    a0, a1 = LZ_PARTS["test"]
    num, den = np.zeros((a1 - a0, Y_seg.shape[2])), np.zeros((a1 - a0, 1))
    for a, y, w in zip(st_test - a0, Y_seg, W):
        num[a:a + SEG_LEN] += w[:, None] * y
        den[a:a + SEG_LEN] += w[:, None]
    return num / den

def lfads_step_time(hp, device, n_steps=30, n_seg=256, seed=0):
    """Millisecondi per passo di allenamento (mediana dopo 5 passi di riscaldamento), su dati finti della forma di hp."""
    import time
    import numpy as np                                      # numpy PRIMA di torch
    import torch
    from lfads_torch.tuples import SessionBatch
    torch.set_num_threads(1); torch.manual_seed(seed)
    model = make_lfads(hp)
    model.__class__ = type("LFADSconEpoca", (type(model),), {"current_epoch": 0})
    model = model.to(device); model.train()
    model.log = model.log_dict = lambda *a, **k: None
    opt = model.configure_optimizers()
    X = torch.tensor(np.random.default_rng(seed).poisson(0.2, size=(n_seg, hp["seg_len"], hp["n_out"])), dtype=torch.float32, device=device)
    times = []
    for s_ in range(n_steps):
        idx = torch.randint(0, n_seg, (64,), device=device)
        sb = SessionBatch(X[idx][..., :hp["n_in"]], X[idx], torch.zeros(64, hp["seg_len"], 0, device=device), torch.zeros(0), torch.ones_like(X[idx]))
        t0 = time.time()
        loss = model._shared_step({0: (sb, ())}, s_, "train")
        opt.zero_grad(); loss.backward(); opt.step()
        if device == "mps":
            torch.mps.synchronize()                         # sulla GPU i calcoli sono asincroni: si aspetta che finiscano
        times.append(time.time() - t0)
    return float(np.median(times[5:]) * 1000)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 135
def real_job(hp, seed, val_trials, segs, hidden=None, k=0):  # un allenamento sui dati veri nel blocco k: impara sui trial di training tranne val_trials
    fit = [int(t) for t in TRIALS if t not in TEST_TRIALS[k] and t not in val_trials]  # i 10 trial su cui impara
    hid = [] if hidden is None else [hidden]                # le colonne nascoste all'encoder (se ci sono)
    return dict(hp=hp, seed=seed, train=np.concatenate([segs[t] for t in fit]),
                val=np.concatenate([segs[t] for t in val_trials]),
                apply=[segs[t] for t in TRIALS],             # le uscite di tutti i 15 trial (quelle di test servono al §12)
                watch={"trial visti (4, 5)": (np.concatenate([segs[t] for t in CHECK_TRIALS]), None, *hid),
                       "validazione": (np.concatenate([segs[t] for t in val_trials]), None, *hid)})

rows11 = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 138
def bits_per_spike(rates, X):                               # bit per spike: guadagno sulla frequenza media di ogni canale, in questi dati (§9.4)
    k, lam = X.reshape(-1, X.shape[-1]), rates.reshape(-1, X.shape[-1])
    lam0 = np.maximum(k.mean(0), 1e-9)                      # la frequenza media di ogni canale (mai esattamente 0)
    return float((np.sum(lam0 - k * np.log(lam0)) - np.sum(lam - k * np.log(lam))) / k.sum() / np.log(2))

def smooth_glm_rates(fit_trials, alpha, C=None):            # frequenze dei canali nascosti da una regressione di Poisson sui visibili smussati, stimata sui trial fit_trials
    C = COUNTS if C is None else C                          # i conteggi da predire (al §13.4 quelli spostati nel tempo)
    fit_ = np.isin(trial, fit_trials)
    F = StandardScaler().fit(SM_VIS[fit_]).transform(SM_VIS)
    return np.column_stack([PoissonRegressor(alpha=alpha, max_iter=500).fit(F[fit_], C[fit_, h]).predict(F) for h in HIDDEN])

rows_ref = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 142
def lfads_sample_prior(n, w, hp, seed=0):
    """Pezzi finti: condizione iniziale estratta dalla distribuzione a priori, poi la seconda metà di lfads_forward_np
    (generatore, fattori, frequenze), poi spike di Poisson. Restituisce (frequenze, spike)."""
    rng_ = np.random.default_rng(seed)
    ic = w["ic_prior.mean"] + np.exp(0.5 * w["ic_prior.logvar"]) * rng_.normal(size=(n, hp["ic_dim"]))  # dalla distribuzione a priori
    g = ic @ w["decoder.ic_to_g0.weight"].T + w["decoder.ic_to_g0.bias"]  # lo stato di partenza del generatore
    W_fac = w["decoder.rnn.cell.fac_linear.weight"]
    W_fac = W_fac / np.linalg.norm(W_fac, axis=1, keepdims=True)  # direzioni dei fattori normalizzate, come nella libreria
    lam = []
    for _ in range(hp["seg_len"]):                          # 30 bin, solo le regole del generatore
        g = gru_step(np.zeros((n, 0)), g, w, "decoder.rnn.cell.gen_cell.", hp["cell_clip"])
        lam.append(np.exp((g @ W_fac.T) @ w["readout.0.weight"].T + w["readout.0.bias"]))  # fattori -> frequenze
    lam = np.stack(lam, 1)
    return lam, rng_.poisson(lam).astype(np.float32)

def pair_corr(X):                                           # correlazioni tra coppie di canali, sugli spike sommati in finestre da 5 bin (100 ms) dentro i pezzi
    Y = X.reshape(len(X), SEG_LEN // 5, 5, X.shape[2]).sum(2).reshape(-1, X.shape[2])
    return np.corrcoef(Y.T)[np.triu_indices(X.shape[2], 1)]

def pop_autocorr(X, lags=range(1, 11)):                     # autocorrelazione nel tempo degli spike della popolazione, dentro i pezzi
    P = X.sum(2)                                            # spike di tutta la popolazione, bin per bin: (pezzi, 30)
    P = P - P.mean()
    return np.array([np.sum(P[:, :-l] * P[:, l:]) / np.sum(P[:, :-l] ** 2) for l in lags])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 143
cols11 = {"veri": "k", "finti (a priori)": "C3", "dalle frequenze a posteriori": "C0"}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 145
def welch_pop(mask, L=50, step=10):                         # spettro medio degli spike della popolazione su finestre da L bin tutte dentro `mask`
    starts = [a for a in range(0, n_bins - L, step) if mask[a:a + L].all()]
    win = np.hanning(L)                                     # una finestra che si annulla ai bordi (attenua le frequenze spurie)
    S = np.mean([np.abs(np.fft.rfft((P_pop[a:a + L] - P_pop[a:a + L].mean()) * win)) ** 2 for a in starts], axis=0)
    return np.fft.rfftfreq(L, io.BIN_S), S, len(starts)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 150
GRID_DROP, GRID_KL, GRID_CD = (0.05, 0.25, 0.5), (1, 5), (0.0, 0.3)

CENTER = (0.05, 1, 0.3)

def grid_hp(drop, kl, cd):                                  # i parametri di una configurazione della griglia (forma "120 + 40")
    return dict(HID_HP, dropout=drop, kl_ic_scale=kl / (SEG_LEN * N_CH), cd_rate=cd)

jobs_g, keys_g = [], []

SMOOTH_EP = 21

def smooth_curve(h, col):                                   # una colonna della storia, smussata su 21 epoche
    return uniform_filter1d(h[col].values, SMOOTH_EP, mode="nearest")

def grid_row(cfg):                                          # le misure di una configurazione: massimo della curva smussata, epoca, trial visti, per ogni coppia
    row, vals, eps = {}, [], []
    for vt, _ in PAIRS_11:
        r_ = res_g[(cfg, vt)]
        h_ = r_["history"]
        v_ = smooth_curve(h_, "bit per spike nascosti, validazione")
        i_ = int(np.argmax(v_))                             # l'epoca del massimo (posizione nella storia)
        lab = f"{vt[0]}-{vt[1]}"
        row[f"nascosti, val. {lab}"] = v_[i_]
        row[f"epoca, val. {lab}"] = int(h_.index[i_])
        row[f"visti (4, 5), val. {lab}"] = smooth_curve(h_, "bit per spike nascosti, trial visti (4, 5)")[i_]
        row[f"epoche fatte, val. {lab}"] = r_["epochs"]
        vals.append(v_[i_]); eps.append(int(h_.index[i_]))
    row["punteggio"] = np.mean(vals)
    row["epoca di picco (media)"] = np.mean(eps)
    return row

def cfg_name(c):                                            # il nome di una configurazione
    return f"dropout {c[0]}, KL × {c[1]}, CD {c[2]}"

def grid_table(cfgs):                                       # la tabella di un gruppo di configurazioni, dal punteggio più alto
    t_ = pd.DataFrame({cfg_name(c): grid_row(c) for c in cfgs}).T
    return t_.sort_values("punteggio", ascending=False)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 152
GRID2 = [(0.0, 0.3, 0.0), (0.0, 1, 0.0), (0.02, 0.3, 0.0), (0.02, 1, 0.0), (0.05, 0.3, 0.0)]

jobs_g2, keys_g2 = [], []

In [ ]:
# REPLICA: solo le definizioni della cella tolta 156
from sklearn.decomposition import FactorAnalysis

import pickle

def train_trials(k):                                        # i 12 trial di training del blocco k (k = "tutti": tutti i 15)
    return [int(t) for t in TRIALS] if k == "tutti" else [int(t) for t in TRIALS if t not in TEST_TRIALS[k]]

def lf12_job(seed, trs, hp=None, segs=None, rate_cols=None):  # LFADS sui trial trs per N_EP_LF epoche; proietta tutti i 15 trial
    tr_ = np.concatenate([(SEGS if segs is None else segs)[t] for t in trs])
    return dict(hp=LF_HP if hp is None else hp, seed=seed, train=tr_, val=tr_,  # insieme di controllo = i pezzi di training (si tiene in pratica l'ultima versione)
                apply=[(SEGS if segs is None else segs)[t] for t in TRIALS], max_epochs=N_EP_LF, patience=N_EP_LF,
                rate_cols=KEEP_CH if rate_cols is None else rate_cols)

def order_by_readout(F, W):                                 # fattori ordinati per importanza: SVD della matrice d'uscita W (canali × fattori)
    U_, S_, Vt_ = np.linalg.svd(W, full_matrices=False)
    return F @ Vt_.T * S_

def stitch_all(per_trial):                                  # ricuce le uscite di tutti i 15 trial e le mette in fila: 12233 bin
    return np.concatenate([stitch(y_, t) for t, y_ in zip(TRIALS, per_trial)])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 157
GP_ITERS = 500

def gpfa_fit_whole(segs, x_dim, iters):
    """GPFA sui pezzi interi (senza il taglio a 20 bin di elephant, come al 02_new §10.6). Gira in un processo separato."""
    import contextlib, os                                   # per silenziare i messaggi di elephant
    import numpy as np
    from elephant.gpfa import gpfa_core
    cut_orig = gpfa_core.gpfa_util.cut_trials               # la funzione che taglia i pezzi...
    gpfa_core.gpfa_util.cut_trials = lambda seqs, seg_length=np.inf: cut_orig(seqs, seg_length=np.inf)  # ...sostituita da una che non taglia
    try:
        np.random.seed(0)                                   # punto di partenza fisso
        seqs = np.array([(s_.shape[0], np.sqrt(s_.astype(float)).T) for s_ in segs], dtype=[("T", int), ("y", "O")])  # radice dei conteggi
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
            params, _ = gpfa_core.fit(seqs, x_dim=x_dim, bin_width=20.0, em_max_iters=iters, tau_init=100.0)
    finally:
        gpfa_core.gpfa_util.cut_trials = cut_orig           # in ogni caso si rimette la funzione originale
    return params

def gp_key(segs, x_dim, iters):                             # impronta di una stima della GPFA: codice, impostazioni, dati
    h = hashlib.sha1(inspect.getsource(gpfa_fit_whole).encode())
    h.update(json.dumps([x_dim, iters]).encode()); h.update(np.ascontiguousarray(segs).tobytes())
    return h.hexdigest()[:16]

def gpfa_cached(seg_sets, x_dim=K_PCA, iters=GP_ITERS):     # le GPFA di più insiemi di pezzi: rilette dal disco o stimate 3 alla volta
    keys = [gp_key(sg, x_dim, iters) for sg in seg_sets]
    todo = [i for i, k_ in enumerate(keys) if RETRAIN or not (CACHE_GP / f"{k_}.pkl").exists()]
    fits = Parallel(n_jobs=3)(delayed(gpfa_fit_whole)(seg_sets[i], x_dim, iters) for i in todo)
    for i, p_ in zip(todo, fits):
        with open(CACHE_GP / f"{keys[i]}.pkl", "wb") as f_:
            pickle.dump(p_, f_)
    print(f"GPFA: {len(todo)} stimate ora, {len(keys) - len(todo)} rilette dal disco")
    out = []
    for k_ in keys:
        with open(CACHE_GP / f"{k_}.pkl", "rb") as f_:
            out.append(pickle.load(f_))
    return out

def fa_factors(trs):                                        # FA con 25 fattori sulla radice dei conteggi dei trial trs (tutti i loro bin); fattori ordinati di tutta la sessione
    Y = np.sqrt(COUNTS.astype(float))
    fa_ = FactorAnalysis(n_components=K_PCA, svd_method="lapack", random_state=0).fit(Y[np.isin(trial, trs)])
    return order_by_readout(fa_.transform(Y), fa_.components_.T)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 159
def run_measure(name, meas):                                # le accuratezze di un modello per una misura: una per seme (5 per UMAP anche con un solo modello)
    gs = GETTERS12[name]
    if meas == "UMAP, d = 3":                               # UMAP in parallelo: ai processi passano solo i 1106 punti di ogni blocco, non tutta la sessione
        inps = [fold_inp(g) for g in gs] if len(gs) > 1 else [fold_inp(gs[0])] * 5
        return Parallel(n_jobs=5)(delayed(decode_map)(inp_, emb_umap(3, seed=i)) for i, inp_ in enumerate(inps))
    return [MEASURES12[meas](g, i) for i, g in enumerate(gs)]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 160
def decode_subset(get, keep, d=None):                       # il metro comune solo sui punti `keep` (training e test), con i primi d fattori (None = tutti)
    idx = np.flatnonzero(keep)
    pred = pd.Series(index=idx, dtype=object)
    for k, (trn, tst) in enumerate(FOLDS):
        trn_k, tst_k = trn[keep[trn]], tst[keep[tst]]       # punti di training e di test del blocco k, senza SnapTo
        F = get(k) if d is None else get(k)[:, :d]
        clf = KNeighborsClassifier(n_neighbors=K_NN).fit(F[bins[trn_k]], phase[bins[trn_k]])
        pred.loc[tst_k] = clf.predict(F[bins[tst_k]])
    return balanced_accuracy_score(phase[bins[idx]], pred.loc[idx].values) * 100

rows_ns = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 161
MEAS_PLOT = ["primi 2 fattori", "primi 3 fattori", "PCA, d = 3", "Isomap, d = 3", "UMAP, d = 3", "tetto: tutti i 25 fattori"]

COL12 = {"FA": "C1", "GPFA come nel §7b": "C4", "GPFA alla pari": "C0", "LFADS": "C3"}

w12 = 0.2

In [ ]:
# REPLICA: solo le definizioni della cella tolta 163
def traj_of(F):                                             # le misure del §4.2 con i primi 3 fattori ordinati
    return traj_stats([w_[:, :3] for w_ in cut(F)])

def fmt3(v):                                                # "media (minimo-massimo)" con 3 decimali (fmt della parte A ne ha uno: è per le percentuali)
    v = np.atleast_1d(v)
    return f"{v.mean():.3f}" if len(v) == 1 else f"{v.mean():.3f} ({v.min():.3f}-{v.max():.3f})"

rows_t12 = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 167
def rough_split(F):                                         # seghettatura dei primi 3 fattori nei bin delle zone comuni e nei bin di un pezzo solo
    d2 = np.concatenate([(np.diff((T_ - T_.mean(0)) / (T_.std(0) + 1e-9), n=2, axis=0) ** 2).mean(1)  # come roughness (§4.2): un valore per bin centrale
                         for T_ in (F[w["start"]:w["stop"], :3] for w in windows)])
    return d2[IN_OVL[MID]].mean(), d2[~IN_OVL[MID]].mean()

def share_18(F, L=50, step=25):                             # quota della potenza dei primi 3 fattori tra 16 e 20 Hz, spettro medio su finestre da 1 s
    win = np.hanning(L)                                     # la stessa finestra del §11.3
    S = np.mean([np.abs(np.fft.rfft((F[a:a + L, :3] - F[a:a + L, :3].mean(0)) * win[:, None], axis=0)) ** 2
                 for a in range(0, n_bins - L, step)], axis=0)  # potenza per frequenza, una colonna per fattore
    f_ = np.fft.rfftfreq(L, io.BIN_S)                       # le frequenze: da 0 a 25 Hz ogni 1 Hz
    return S[(f_ >= 16) & (f_ <= 20)].sum(0) / S[1:].sum(0)

def cross_tr(w):                                            # vero se nella finestra w cambia il trial
    return bool((np.diff(trial[w["start"]:w["stop"]]) != 0).any())

rows_c = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 170
from scipy.stats import poisson

from sklearn.linear_model import PoissonRegressor

def test_only(per_block):                                   # da "blocco -> previsione di tutta la sessione" alla sessione fatta solo di trial di test
    out = np.empty_like(per_block[0])
    for k in range(5):
        tb = np.isin(trial, TEST_TRIALS[k])                 # i bin dei 3 trial di test del blocco k
        out[tb] = per_block[k][tb]                          # presi dal modello del blocco k, che non li ha mai visti
    return out

In [ ]:
# REPLICA: solo le definizioni della cella tolta 171
def gpfa_vis(segs, params, ch):                             # i fattori della GPFA ricavati dai soli canali ch (media e incertezza), pezzo per pezzo
    p_ = dict(params, C=params["C"][ch], d=params["d"][ch], R=params["R"][np.ix_(ch, ch)])  # il modello ristretto ai canali ch
    with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
        lat, _ = gpfa_core.exact_inference_with_ll(gpfa_seqs(segs[..., ch]), p_, get_ll=False)
    return lat["latent_variable"], lat["Vsm"]

def gpfa_hidden(segs, params):                              # conte attese dei nascosti in ogni pezzo, dai soli visibili: completa e variante μ² + R
    X_, V_ = gpfa_vis(segs, params, HELD_IN)
    C_h, d_h, r_h = params["C"][HIDDEN], params["d"][HIDDEN], np.diag(params["R"])[HIDDEN]  # pesi, medie e rumore dei nascosti
    mu = np.stack([(C_h @ x_ + d_h[:, None]).T for x_ in X_])  # media della radice dei conteggi: (pezzi, bin, 40)
    unc = np.stack([np.einsum("hi,ijt,hj->th", C_h, v_, C_h) for v_ in V_])  # incertezza sui fattori portata su ogni canale: c Σ cᵀ
    return mu ** 2 + r_h + unc, mu ** 2 + r_h

def fa_hidden(trs, C=None):                                 # FA sui 160 canali dei trial trs; conte attese dei nascosti in tutta la sessione, dai soli visibili
    Y = np.sqrt((COUNTS if C is None else C).astype(float))  # i conteggi (al §13.4 quelli spostati nel tempo)
    fa_ = FactorAnalysis(n_components=K_PCA, svd_method="lapack", random_state=0).fit(Y[np.isin(trial, trs)])
    L, psi, m = fa_.components_.T, fa_.noise_variance_, fa_.mean_  # pesi (160 × 25), rumore di ogni canale (Ψ), medie
    Lv, Lh = L[HELD_IN], L[HIDDEN]
    S = np.linalg.inv(np.eye(K_PCA) + Lv.T @ (Lv / psi[HELD_IN, None]))  # incertezza sui fattori, dati i 120 visibili (Σ)
    X = ((Y[:, HELD_IN] - m[HELD_IN]) / psi[HELD_IN]) @ Lv @ S  # fattori di ogni bin dai soli visibili: Σ Λᵀ Ψ⁻¹ (y − m)
    mu = X @ Lh.T + m[HIDDEN]                               # media della radice dei conteggi dei nascosti
    return mu ** 2 + psi[HIDDEN] + np.einsum("hi,ij,hj->h", Lh, S, Lh)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 172
def gpfa_glm(k, segs=None, gp=None, C=None, trs=None):     # variante: regressione di Poisson dai 25 fattori GPFA (ricavati dai visibili) ai nascosti, nel blocco k
    segs, gp, C = (SEGS if segs is None else segs), (GP12 if gp is None else gp), (COUNTS if C is None else C)  # al §13.4: i dati spostati nel tempo
    lat = np.concatenate([stitch(np.stack([x_.T for x_ in gpfa_vis(segs[t], gp[k], HELD_IN)[0]]), t) for t in TRIALS])  # fattori di tutta la sessione
    fit_ = np.isin(trial, train_trials(k) if trs is None else trs)  # la regressione si stima sui 12 trial di training (al §13b su meno trial)
    F = StandardScaler().fit(lat[fit_]).transform(lat)
    return np.column_stack([PoissonRegressor(alpha=0.1, max_iter=500).fit(F[fit_], C[fit_, h]).predict(F) for h in HIDDEN])

def fmt4(v):                                                # "media [minimo, massimo]" con 4 decimali (le parentesi quadre perché i valori possono essere negativi)
    v = np.atleast_1d(v)
    return f"{v.mean():.4f}" if len(v) == 1 else f"{v.mean():.4f} [{v.min():.4f}, {v.max():.4f}]"

BPS12 = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 174
MAIN12B = ["smoothing + GLM (riferimento)", "FA", "GPFA alla pari", "LFADS (5 semi)"]

COL12B = {"smoothing + GLM (riferimento)": "0.45", "FA": "C1", "GPFA alla pari": "C0", "LFADS (5 semi)": "C3"}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 179
cons12 = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 181
def seen_new(full, Y):                                      # bit per spike sui trial visti e su quelli nuovi, blocco per blocco: (semi, 5 blocchi, 2)
    out = np.zeros((len(full[0]), 5, 2))
    for k in range(5):
        trn_b, tst_b = np.isin(trial, train_trials(k)), np.isin(trial, TEST_TRIALS[k])  # i bin dei 12 trial di training e dei 3 di test
        for i, p_ in enumerate(full[k]):
            out[i, k] = bits_per_spike(p_[trn_b], Y[trn_b]), bits_per_spike(p_[tst_b], Y[tst_b])
    return out

In [ ]:
# REPLICA: solo le definizioni della cella tolta 184
def poisson_twin(seed):                                     # gemello di Poisson: radice delle frequenze smussate a fasi casuali, al quadrato, poi conteggi a caso
    R_tw = phase_randomize(np.sqrt(SM_RATE), seed) ** 2     # frequenze del gemello (Hz), sempre positive
    return np.random.default_rng(1000 + seed).poisson(R_tw * io.BIN_S).astype(np.float32), R_tw

In [ ]:
# REPLICA: solo le definizioni della cella tolta 187
REAL_NAME = {"LFADS (3 semi)": "LFADS (5 semi)", "GPFA + regressione di Poisson": "variante: GPFA + regressione di Poisson"}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 190
from itertools import zip_longest

SIZES13 = (4, 7, 10, 12)

def trial_order(k):                                         # l'ordine in cui si aggiungono i 12 trial di training del blocco k
    rng_ = np.random.default_rng(100 + k)                   # seme fisso, diverso per ogni blocco
    odd = [int(t) for t in rng_.permutation([t for t in train_trials(k) if t % 2 == 1])]  # i dispari, in ordine casuale
    even = [int(t) for t in rng_.permutation([t for t in train_trials(k) if t % 2 == 0])]  # i pari, in ordine casuale
    first, second = (odd, even) if k % 2 == 0 else (even, odd)  # nei blocchi 0, 2, 4 si comincia da un dispari, negli altri da un pari
    return [t for pair in zip_longest(first, second) for t in pair if t is not None]

def subset(k, n):                                           # i primi n trial di quell'ordine, rimessi in ordine di tempo (n = 12: train_trials(k))
    return sorted(TRIAL_ORDER[k][:n])

def n_batch(trs):                                           # gruppi da 64 pezzi in un'epoca di allenamento (l'ultimo può essere più piccolo)
    return int(np.ceil(n_seg(trs) / 64))

def ep_same_steps(k, n):                                    # le epoche con cui n trial fanno almeno i passi del §12 (200 epoche con i 12 trial)
    return int(np.ceil(N_EP_LF * n_batch(train_trials(k)) / n_batch(subset(k, n))))

def mm(v):                                                  # "minimo-massimo" di una serie di numeri interi (un numero solo se sono tutti uguali)
    return f"{min(v)}" if min(v) == max(v) else f"{min(v)}-{max(v)}"

In [ ]:
# REPLICA: solo le definizioni della cella tolta 191
def job13(form, n, k, sd, rule):                            # un allenamento: forma, quanti trial, blocco, seme, regola
    j_ = lf12_job(sd, subset(k, n), **FORMS13[form])        # come al §12, ma sui trial ridotti
    if rule == "passi del §12":                             # il controllo: più epoche, finché i passi sono quelli del §12
        j_.update(max_epochs=ep_same_steps(k, n), patience=ep_same_steps(k, n))
    return j_

In [ ]:
# REPLICA: solo le definizioni della cella tolta 193
GET13 = {}

NAME12 = {"FA": "FA", "GPFA alla pari": "GPFA alla pari", "LFADS (3 semi)": "LFADS"}

def fmt1s(v):                                               # "media [minimo, massimo]" con un decimale (le parentesi quadre perché può essere negativo)
    v = np.atleast_1d(v)
    return f"{v.mean():.1f}" if len(v) == 1 else f"{v.mean():.1f} [{v.min():.1f}, {v.max():.1f}]"

NAMES_PH13 = ["FA", "GPFA alla pari", "LFADS (3 semi)", "LFADS, passi del §12"]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 195
def gpfa_hidden_test(gp_of):                                # conte attese dei nascosti di ogni trial di test, dalla GPFA del suo blocco (gp_of: blocco -> parametri)
    pieces = {int(t): gpfa_hidden(SEGS[t], gp_of(k))[0] for k in range(5) for t in TEST_TRIALS[k]}
    return np.concatenate([stitch(pieces[int(t)], t) for t in TRIALS])

PRED13 = {}

NAME12B = {"smoothing + GLM (riferimento)": "smoothing + GLM (riferimento)", "FA": "FA", "GPFA alla pari": "GPFA alla pari",
           "GPFA + regressione di Poisson": "variante: GPFA + regressione di Poisson", "LFADS (3 semi)": "LFADS (5 semi)"}

def bps_blocks(P):                                          # bit per spike sui 3 trial di test di ogni blocco: (semi, 5 blocchi)
    return np.array([[bits_per_spike(p_[np.isin(trial, TEST_TRIALS[k])], Y_HID[np.isin(trial, TEST_TRIALS[k])]) for k in range(5)] for p_ in P])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 196
COL13 = {"smoothing + GLM (riferimento)": "0.45", "FA": "C1", "GPFA alla pari": "C0", "GPFA + regressione di Poisson": "C2", "LFADS (3 semi)": "C3"}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 200
def job14(var, form, k, sd):                                # un allenamento: variante, forma, blocco, seme; il resto come al §12
    hp_, data_ = FORMS14[form]
    return lf12_job(sd, train_trials(k), hp=dict(hp_, **VARIANTS14A[var]), **data_)

def shapes(w):                                              # le grandezze di un modello lette dai suoi pesi: fattori, generatore
    return w["readout.0.weight"].shape[1], w["decoder.rnn.cell.gen_cell.weight_hh"].shape[1]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 202
def acc14(get):                                             # fase riconosciuta: (tetto, primi 3) per una funzione blocco -> fattori
    return decode_map(fold_inp(get), emb_none), decode_map(fold_inp(lambda k: get(k)[:, :3]), emb_none)

def full_lf14(var):                                         # [blocco][seme] -> conte attese dei 40 nascosti in tutta la sessione (forma 120 + 40)
    return [[stitch_all(res14a[(var, "120 + 40", k, sd)]["rates"]) for sd in SEEDS14] for k in range(5)]

def full_gp14(d_):                                          # lo stesso per la GPFA alla pari con d_ fattori (un solo modello per blocco)
    if d_ == K_PCA:
        return FULL13["GPFA alla pari"]                     # con 25 fattori: già calcolata al §13.2
    return [[np.concatenate([stitch(gpfa_hidden(SEGS[t], GP14[d_][k])[0], t) for t in TRIALS])] for k in range(5)]

def fmt2(v):                                                # "media [minimo, massimo]" con 2 decimali
    v = np.atleast_1d(v)
    return f"{v.mean():.2f}" if len(v) == 1 else f"{v.mean():.2f} [{v.min():.2f}, {v.max():.2f}]"

M14 = {}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 206
LENS14 = {"pezzi da 400 ms": 20, "pezzi da 1 s": 50}

def starts_L(t, L):                                         # come segment_starts (§9.2), per pezzi di L bin che si sovrappongono di 10
    a0, a1 = TRIAL_SPAN[t]
    st = list(range(a0, a1 - L + 1, L - OVL14))             # un pezzo ogni L − 10 bin...
    if st[-1] != a1 - L:
        st.append(a1 - L)                                   # ...più un ultimo pezzo che finisce esattamente con il trial
    return np.array(st)

def cut_L(X, t, L):                                         # come cut_segments: i pezzi da L bin del trial t di un segnale X
    return np.stack([X[a:a + L] for a in starts_L(t, L)])

def stitch_L(Y_seg, t, L):                                  # come stitch: media pesata, con le rampe di AutoLFADS sui 10 bin in comune
    a0, a1 = TRIAL_SPAN[t]
    W = np.ones((len(Y_seg), L))                            # 1 dove il pezzo è da solo
    W[1:, :OVL14] = RAMP_IN                                 # rampa in salita all'inizio (tranne il primo pezzo del trial)
    W[:-1, -OVL14:] = RAMP_OUT                              # rampa in discesa alla fine (tranne l'ultimo)
    num, den = np.zeros((a1 - a0, Y_seg.shape[2])), np.zeros((a1 - a0, 1))
    for a, y, w in zip(starts_L(t, L) - a0, Y_seg, W):
        num[a:a + L] += w[:, None] * y                      # somma dei valori pesati...
        den[a:a + L] += w[:, None]                          # ...e dei pesi
    return num / den

def stitch_all_L(per_trial, L):                             # come stitch_all: i 15 trial ricuciti e messi in fila
    return np.concatenate([stitch_L(y_, t, L) for t, y_ in zip(TRIALS, per_trial)])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 207
def make_lfads_co(hp):
    """Come make_lfads (§10.3), ma con il controller: un secondo encoder, il controller e 4 ingressi inferiti. Si chiama nel processo separato."""
    import numpy as np                                      # numpy PRIMA di torch (§10.3)
    import torch
    from torch import nn
    from lfads_torch.model import LFADS                     # il modello della libreria
    from lfads_torch.modules import augmentations           # il coordinated dropout (qui a 0, come al §12)
    from lfads_torch.modules.priors import AutoregressiveMultivariateNormal, MultivariateNormal  # priori degli ingressi e della condizione iniziale
    from lfads_torch.modules.recons import Poisson          # il modello del rumore: Poisson
    from lfads_torch.modules.readin_readout import FanInLinear  # l'uscita lineare dai fattori alle log-frequenze
    cd = augmentations.CoordinatedDropout(cd_rate=hp["cd_rate"], cd_pass_rate=0.0, ic_enc_seq_len=0)
    return LFADS(encod_data_dim=hp["n_in"], encod_seq_len=hp["seg_len"], recon_seq_len=hp["seg_len"],  # forma dei dati
                 ext_input_dim=0, ic_enc_seq_len=0,         # nessun ingresso esterno; l'encoder legge tutto il pezzo
                 ic_enc_dim=hp["enc_dim"], ci_enc_dim=hp["ci_enc_dim"], ci_lag=1, con_dim=hp["con_dim"], co_dim=hp["co_dim"],  # il controller è acceso
                 ic_dim=hp["ic_dim"], gen_dim=hp["gen_dim"], fac_dim=hp["fac_dim"], dropout_rate=hp["dropout"],
                 reconstruction=nn.ModuleList([Poisson()]), variational=True,
                 co_prior=AutoregressiveMultivariateNormal(tau=hp["co_tau"], nvar=hp["co_nvar"], shape=hp["co_dim"]),  # priori degli ingressi
                 ic_prior=MultivariateNormal(mean=0.0, variance=hp["ic_prior_var"], shape=hp["ic_dim"]),
                 ic_post_var_min=hp["ic_post_var_min"], cell_clip=hp["cell_clip"],
                 train_aug_stack=augmentations.AugmentationStack([cd], batch_order=[0], loss_order=[0]),
                 infer_aug_stack=augmentations.AugmentationStack(),
                 readin=nn.ModuleList([nn.Identity()]), readout=nn.ModuleList([FanInLinear(hp["fac_dim"], hp["n_out"])]),
                 loss_scale=hp["loss_scale"], recon_reduce_mean=True,
                 lr_scheduler=False, lr_init=hp["lr"], lr_stop=1e-5, lr_decay=0.95, lr_patience=6,
                 lr_adam_beta1=0.9, lr_adam_beta2=0.999, lr_adam_epsilon=1e-8, weight_decay=0.0,
                 l2_start_epoch=0, l2_increase_epoch=hp["ramp_epochs"], l2_ic_enc_scale=hp["l2_ic_enc_scale"],
                 l2_ci_enc_scale=hp["l2_ci_enc_scale"], l2_gen_scale=hp["l2_gen_scale"], l2_con_scale=hp["l2_con_scale"],
                 kl_start_epoch=0, kl_increase_epoch=hp["ramp_epochs"], kl_ic_scale=hp["kl_ic_scale"], kl_co_scale=hp["kl_co_scale"])

def run_lfads_cached_co(jobs):                              # run_lfads_cached (§10.5) con il modello del controller
    global make_lfads                                       # durante la chiamata il nome make_lfads indica make_lfads_co: così lfads_fit,
    orig = make_lfads                                       # nel processo separato, costruisce il modello con il controller, e l'impronta
    make_lfads = make_lfads_co                              # di ogni modello (lf_key) contiene il codice di make_lfads_co
    try:
        return run_lfads_cached(jobs)
    finally:
        make_lfads = orig

def co_prior_check(tau, nvar, seed):                        # log-probabilità di alcune sequenze sotto la priori autoregressiva della libreria (processo separato)
    import numpy as np                                      # numpy PRIMA di torch
    import torch
    from lfads_torch.modules.priors import AutoregressiveMultivariateNormal
    torch.manual_seed(seed)
    prior = AutoregressiveMultivariateNormal(tau=tau, nvar=nvar, shape=4)  # 4 ingressi, come in CO14
    x = torch.randn(3, 30, 4)                               # 3 pezzi da 30 bin, 4 ingressi: valori a caso
    with torch.no_grad():
        return x.numpy().astype(np.float64), prior.log_prob(x).numpy().astype(np.float64)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 208
def job14b(var, form, k, sd):                               # un allenamento del §14b: variante, forma, blocco, seme; il resto come al §12
    hp_, data_ = FORMS14[form]                              # configurazione e dati delle due forme (§14a)
    if var == "controller":
        return lf12_job(sd, train_trials(k), hp=dict(hp_, **CO14), **data_)  # pezzi da 600 ms, con il controller
    L = LENS14[var]
    hp_L = dict(hp_, seg_len=L, kl_ic_scale=hp_["kl_ic_scale"] * SEG_LEN / L)  # peso del KL: 0.3 / (L × 160)
    if form == "tutti i canali":
        return lf12_job(sd, train_trials(k), hp=hp_L, segs=SEGS14[L])
    return lf12_job(sd, train_trials(k), hp=hp_L, segs=SEGS14_HID[L], rate_cols=HID_COLS)

def co_dim_of(w):                                           # quanti ingressi inferiti ha un modello, letto dai pesi (0 = niente controller)
    return next((v.shape[0] // 2 for k_, v in w.items() if k_.endswith("co_linear.weight")), 0)

In [ ]:
# REPLICA: solo le definizioni della cella tolta 210
def fac14b(v, k, sd):                                       # fattori ordinati della variante v (forma con tutti i canali), tutta la sessione
    r = RES14B[(v, "tutti i canali", k, sd)]
    return order_by_readout(stitch_all_L(r["factors"], GEOM14B[v]), r["weights"]["readout.0.weight"])

M14B = {}

GP_SAME14B = {"principale (§12)": "GPFA alla pari, pezzi da 600 ms", "pezzi da 400 ms": "GPFA alla pari, pezzi da 400 ms",
              "pezzi da 1 s": "GPFA alla pari, pezzi da 1 s", "controller": "GPFA alla pari, pezzi da 600 ms"}

In [ ]:
# REPLICA: solo le definizioni della cella tolta 211
def co_inputs(hp, model_path, X):                           # gli ingressi inferiti di un modello con il controller salvato su disco, sui pezzi X (processo separato)
    import numpy as np                                      # numpy PRIMA di torch
    import torch
    from lfads_torch.tuples import SessionBatch
    torch.set_num_threads(1)
    model = make_lfads_co(hp)
    model.load_state_dict(torch.load(model_path))           # i pesi della versione tenuta
    model.eval()                                            # niente dropout
    Xt = torch.tensor(X, dtype=torch.float32)
    sb = SessionBatch(encod_data=Xt[..., :hp["n_in"]], recon_data=Xt, ext_input=torch.zeros(len(Xt), Xt.shape[1], 0),
                      truth=torch.zeros(0), sv_mask=torch.ones_like(Xt))
    with torch.no_grad():
        out = model({0: sb}, sample_posteriors=False, output_means=False)[0]  # con la media a posteriori, come per i fattori
    return out.co_means.numpy(), out.co_stds.numpy()

In [ ]:
# REPLICA: solo le definizioni della cella tolta 214
def job14b_sh(k, sd):                                       # il controller, forma 120 + 40, sui dati con i nascosti spostati nel tempo (§13.4)
    return lf12_job(sd, train_trials(k), hp=dict(LF_HID_HP, **CO14), segs=SEGS_SH_HID, rate_cols=HID_COLS)

def share18_rates(P, L=50, step=25):                        # quota della variazione tra 16 e 20 Hz nelle frequenze previste (media sui canali), finestre da 1 s
    win = np.hanning(L)                                     # la stessa finestra del §11.3
    S = np.mean([np.abs(np.fft.rfft((P[a:a + L] - P[a:a + L].mean(0)) * win[:, None], axis=0)) ** 2 for a in range(0, len(P) - L, step)], axis=0)
    f_ = np.fft.rfftfreq(L, io.BIN_S)                       # da 0 a 25 Hz, ogni 1 Hz
    return float(np.mean(S[(f_ >= 16) & (f_ <= 20)].sum(0) / S[1:].sum(0)))

def glm_sigma(sg, k):                                       # lo smoothing + GLM del §11a con un'altra σ, stimato sui 12 trial di training del blocco k
    fit_ = np.isin(trial, train_trials(k))
    F = StandardScaler().fit(SMV[sg][fit_]).transform(SMV[sg])
    return np.column_stack([PoissonRegressor(alpha=0.1, max_iter=500).fit(F[fit_], COUNTS[fit_, h]).predict(F) for h in HIDDEN])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 217
def job14c(form, k, sd):                                    # un allenamento del VAE per bin: forma, blocco, seme; il resto come al §12
    hp_, _ = FORMS14[form]
    hp1 = dict(hp_, seg_len=1, kl_ic_scale=hp_["kl_ic_scale"] * SEG_LEN)  # pezzi di un bin; peso del KL 0.3 / (1 × 160)
    if form == "tutti i canali":
        j_ = lf12_job(sd, train_trials(k), hp=hp1, segs=SEGS1)
    else:
        j_ = lf12_job(sd, train_trials(k), hp=hp1, segs=SEGS1_HID, rate_cols=HID_COLS)
    j_["apply"] = j_["apply"] + [j_["apply"][list(TRIALS).index(T_SHUF)][PERM]]  # in più: il trial del controllo, rimescolato
    return j_

In [ ]:
# REPLICA: solo le definizioni della cella tolta 218
def fac14c(k, sd):                                          # fattori ordinati del VAE per bin (forma con tutti i canali), tutta la sessione
    r = res14c[("tutti i canali", k, sd)]
    return order_by_readout(np.concatenate([f_[:, 0, :] for f_ in r["factors"][:len(TRIALS)]]), r["weights"]["readout.0.weight"])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 220
def lfads_glm(k, sd):                                       # regressione di Poisson dai 25 fattori di LFADS (forma 120 + 40) ai 40 nascosti, nel blocco k
    lat = stitch_all(res12b[(k, sd)]["factors"])            # i fattori di tutta la sessione, ricavati dai soli 120 visibili
    fit_ = np.isin(trial, train_trials(k))                  # la regressione si stima sui 12 trial di training
    F = StandardScaler().fit(lat[fit_]).transform(lat)
    return np.column_stack([PoissonRegressor(alpha=0.1, max_iter=500).fit(F[fit_], COUNTS[fit_, h]).predict(F) for h in HIDDEN])

def ratio(sn_):                                             # nuovi / visti, seme per seme, da un array (semi, 5 blocchi, 2) di seen_new
    return list(sn_[:, :, 1].mean(1) / sn_[:, :, 0].mean(1))

In [ ]:
# REPLICA: solo le definizioni della cella tolta 221
names_ph = ["FA", "VAE per bin (3 semi)", "GPFA alla pari", "LFADS (principale, semi 0-2)"]

cols_ph = ["C1", "C5", "C0", "C3"]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 224
EXTRA_SID = ("19250108T111022", "19250108T111455")

pieces_ex, inside_ex = [], []

In [ ]:
# REPLICA: solo le definizioni della cella tolta 225
def job14d(form, k, sd):                                    # come al §12, con i pezzi delle sessioni nuove aggiunti ai pezzi di training
    hp_, data_ = FORMS14[form]
    j_ = lf12_job(sd, train_trials(k), hp=hp_, **data_)
    j_["train"] = j_["val"] = np.concatenate([j_["train"], SEGS_EX if form == "tutti i canali" else SEGS_EX_HID])
    return j_

def job14d_co(form, k, sd):                                 # lo stesso, con il controller del §14b
    j_ = job14d(form, k, sd)
    j_["hp"] = dict(j_["hp"], **CO14)
    return j_

In [ ]:
# REPLICA: solo le definizioni della cella tolta 227
def fa_extra(k, extra=True):                                # FA con 25 fattori sui 12 trial di training (e sulle sessioni nuove): fattori ordinati e nascosti
    Y = np.sqrt(COUNTS.astype(float))
    Y_fit = np.concatenate([Y[np.isin(trial, train_trials(k))]] + ([np.sqrt(x_.astype(float)) for x_ in X_EX.values()] if extra else []))
    fa_ = FactorAnalysis(n_components=K_PCA, svd_method="lapack", random_state=0).fit(Y_fit)
    L, psi, m = fa_.components_.T, fa_.noise_variance_, fa_.mean_  # come fa_hidden (§12.4)
    Lv, Lh = L[HELD_IN], L[HIDDEN]
    S = np.linalg.inv(np.eye(K_PCA) + Lv.T @ (Lv / psi[HELD_IN, None]))
    X = ((Y[:, HELD_IN] - m[HELD_IN]) / psi[HELD_IN]) @ Lv @ S  # fattori di ogni bin dai soli visibili
    return order_by_readout(fa_.transform(Y), L), (X @ Lh.T + m[HIDDEN]) ** 2 + psi[HIDDEN] + np.einsum("hi,ij,hj->h", Lh, S, Lh)

def glm_extra(k, extra=True):                               # smoothing + GLM (§11a) stimato sui 12 trial di training (e sulle sessioni nuove)
    fit_ = np.isin(trial, train_trials(k))
    F_fit = np.concatenate([SM_VIS[fit_]] + (SM_VIS_EX if extra else []))
    C_fit = np.concatenate([COUNTS[fit_]] + (list(X_EX.values()) if extra else []))
    sc = StandardScaler().fit(F_fit)
    return np.column_stack([PoissonRegressor(alpha=0.1, max_iter=500).fit(sc.transform(F_fit), C_fit[:, h]).predict(sc.transform(SM_VIS))
                            for h in HIDDEN])

def glm_on(lat, k):                                         # regressione di Poisson da fattori qualsiasi (tutta la sessione) ai nascosti, sui 12 trial di training del blocco k
    fit_ = np.isin(trial, train_trials(k))
    F = StandardScaler().fit(lat[fit_]).transform(lat)
    return np.column_stack([PoissonRegressor(alpha=0.1, max_iter=500).fit(F[fit_], COUNTS[fit_, h]).predict(F) for h in HIDDEN])

In [ ]:
# REPLICA: solo le definizioni della cella tolta 232
def fin_row(lab, a2=None, a3=None, a7=None, tetto=None):    # una riga nuova, nello stesso formato della tabella del §8
    return {"etichette": lab, "2 coordinate libere": fmt(a2) if a2 is not None else "", "3 coordinate libere": fmt(a3) if a3 is not None else "",
            "7 coordinate libere": fmt(a7) if a7 is not None else "", "tutte le dimensioni": fmt(tetto) if tetto is not None else ""}

ORDER15 = ["PCA", "Isomap", "UMAP", "CEBRA-Time (3, 4 e 8 dimensioni)", "FA (primi fattori ordinati)", "GPFA (primi fattori)",
           "GPFA alla pari (primi fattori ordinati)", "LFADS (primi fattori ordinati, 5 semi)", "LFADS con il controller (primi fattori ordinati, 3 semi)",
           "VAE per bin (primi fattori ordinati, 3 semi)",
           "UMAP sui fattori della GPFA alla pari", "UMAP sui fattori di LFADS (5 semi)"]

In [ ]:
# REPLICA: solo le definizioni della cella tolta 233
COL15 = {"PCA": "C1", "Isomap": "C9", "UMAP": "C2", "CEBRA-Time (4 dimensioni)": "C6", "FA": "C1", "VAE per bin": "C5",
         "GPFA alla pari": "C0", "LFADS": "C3", "LFADS con il controller": "C4"}

In [ ]:
# ===== CONTROLLO FASI =====
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
PH = list(io.MOVEMENT_PHASES); K = len(PH)
y = phase[bins].astype(str)

def predict(inp, embed):                                    # come decode_map, ma restituisce le previsioni di ogni punto
    b, folds = inp
    pred = np.empty(len(b), dtype=object)
    for Xtr, Xte, trn, tst in folds:
        Etr, Ete = embed(Xtr, Xte)
        pred[tst] = KNeighborsClassifier(n_neighbors=K_NN).fit(Etr, phase[b[trn]]).predict(Ete)
    return pred.astype(str)

# --- 3. contesto temporale: conteggi grezzi z-scorati, PCA a 25 per blocco (bin di movimento dei trial di training)
Zc = StandardScaler().fit_transform(s.spikes[:, ACTIVE].astype(float))
PCA_C = [PCA(K_PCA).fit(Zc[bins_all[~np.isin(trial[bins_all], tt)]]) for tt in TEST_TRIALS]
XC = [p.transform(Zc) for p in PCA_C]
def causal_mean(X, L):                                      # media dei bin t-L+1 ... t (all'inizio della sessione: quelli che ci sono)
    c = np.cumsum(np.vstack([np.zeros((1, X.shape[1])), X]), axis=0)
    t = np.arange(len(X)); a = np.maximum(t - L + 1, 0)
    return (c[t + 1] - c[a]) / (t + 1 - a)[:, None]
def causal_stack(X, n_sub=10, sub=5):                       # 10 medie causali da 100 ms, dalla più recente: 1000 ms, 250 numeri
    m = causal_mean(X, sub)
    return np.hstack([np.vstack([np.repeat(m[:1], j * sub, 0), m[:len(m) - j * sub]]) for j in range(n_sub)])

clock = np.zeros(n_bins)
for t in TRIALS:
    clock[trial == t] = np.arange((trial == t).sum()) * io.BIN_S

METHODS = {                                                 # nome -> lista di (input, mappa), una per seme
    "PCA, d = 3": [(INP_PC, emb_pca(3))],
    "tetto: 25 PC smussati, 1 bin": [(INP_PC, emb_none)],
    **{f"CEBRA-Time, {od} dimensioni": [(fold_inp(lambda k, od=od, sd=sd: MAPS4[(od, sd, k)]), emb_none) for sd in SEEDS] for od in (3, 4)},
    **{f"CEBRA-Behavior, {od} dimensioni": [(fold_inp(lambda k, od=od, sd=sd: MAPS6[(od, sd, k)]), emb_none) for sd in SEEDS] for od in (3, 4)},
    **{f"PLS, {nc} componenti": [(fold_inp(lambda k, nc=nc: PLS_F[nc][k].transform(X25[k])), emb_none)] for nc in (2, 3)},
    "orologio (solo il tempo dall'inizio del trial)": [(fold_inp(lambda k: clock[:, None]), emb_none)],
    "contesto: 25 PC dei conteggi, 1 bin (causale)": [(fold_inp(lambda k: XC[k]), emb_none)],
    "contesto: 25 PC dei conteggi, media causale 1000 ms": [(fold_inp(lambda k: causal_mean(XC[k], 50)), emb_none)],
    "contesto: 25 PC dei conteggi, 10 × 100 ms causali (250 numeri)": [(fold_inp(lambda k: causal_stack(XC[k])), emb_none)],
    "contesto: 25 PC smussati, 10 × 100 ms causali (250 numeri)": [(fold_inp(lambda k: causal_stack(X25[k])), emb_none)],
}
PRED = {name: [predict(i_, e_) for i_, e_ in lst] for name, lst in METHODS.items()}

def adjacent_share(yt, yp):                                 # quota degli errori che cadono su una fase adiacente nell'ordine del task
    i_t = np.array([PH.index(v) for v in yt]); i_p = np.array([PH.index(v) for v in yp])
    err = i_t != i_p
    obs = np.mean(np.abs(i_t[err] - i_p[err]) == 1)
    n_adj = np.array([(i > 0) + (i < K - 1) for i in range(K)])  # vicine di ogni fase (Reach e Release ne hanno una)
    exp = np.mean(n_adj[i_t[err]] / (K - 1))                # se gli errori fossero sparsi a caso tra le altre 7 fasi
    return obs * 100, exp * 100

counts = pd.Series(y).value_counts()
rows = {}
for name, preds in PRED.items():
    bal = [balanced_accuracy_score(y, p) * 100 for p in preds]
    acc = [np.mean(y == p) * 100 for p in preds]
    adj = np.array([adjacent_share(y, p) for p in preds])
    rows[name] = {"semi": len(preds), "accuratezza bilanciata (%)": mean_range(bal) if len(bal) > 1 else f"{bal[0]:.1f}",
                  "accuratezza semplice (%)": mean_range(acc) if len(acc) > 1 else f"{acc[0]:.1f}",
                  "errori su fasi adiacenti (%)": f"{adj[:, 0].mean():.0f}",
                  "se sparsi a caso (%)": f"{adj[:, 1].mean():.0f}"}
TAB_FASI = pd.DataFrame(rows).T
TAB_FASI.loc["caso: 1/K"] = ["", f"{100 / K:.1f}", "", "", ""]
TAB_FASI.loc["caso: classe più frequente"] = ["", f"{100 / K:.1f}", f"{counts.max() / len(y) * 100:.1f} ({counts.idxmax()})", "", ""]
display(TAB_FASI)
print(f"Punti di valutazione: {len(y)}; per fase: {counts.reindex(PH).to_dict()}")
print("Nota: la balanced accuracy di un classificatore che dice sempre la classe più frequente è 1/K = 12.5%.")

# --- 2. matrici di confusione (righe: fase vera, normalizzate; somma sui semi)
SHOW = ["CEBRA-Time, 4 dimensioni", "CEBRA-Behavior, 4 dimensioni", "PLS, 3 componenti",
        "orologio (solo il tempo dall'inizio del trial)", "tetto: 25 PC smussati, 1 bin",
        "contesto: 25 PC dei conteggi, 10 × 100 ms causali (250 numeri)"]
fig, ax = plt.subplots(2, 3, figsize=(17, 10.5))
CM = {}
for a_, name in zip(ax.ravel(), SHOW):
    cm = sum(confusion_matrix(y, p, labels=PH) for p in PRED[name]).astype(float)
    cm = cm / cm.sum(1, keepdims=True) * 100; CM[name] = cm
    a_.imshow(cm, cmap="Blues", vmin=0, vmax=100)
    for i in range(K):
        for j in range(K):
            a_.text(j, i, f"{cm[i, j]:.0f}", ha="center", va="center", fontsize=7.5, color="w" if cm[i, j] > 55 else "k")
    a_.set(xticks=range(K), yticks=range(K), xticklabels=PH, yticklabels=PH, title=name[:60])
    a_.tick_params(axis="x", rotation=45, labelsize=8); a_.tick_params(axis="y", labelsize=8); a_.grid(False)
fig.supxlabel("fase prevista"); fig.supylabel("fase vera (righe in %)")
fig.tight_layout(); fig.savefig(FIGS / "fasi_confusione.png", bbox_inches="tight"); plt.show()

import pickle
pickle.dump(dict(TAB_FASI=TAB_FASI, CM=CM, PRED=PRED, y=y, PH=PH), open(FIGS.parent / "fasi_04.pkl", "wb"))
